# QE-XAI — small-data, multi-part edition: every classifier on several small parts of OASIS-1 (CN vs AD)

This is a revised version of the notebook *oasis-new-version-17-oct* (QE-XAI: hybrid CNN–VQC for Alzheimer's disease detection on OASIS-1). The pre-processing, the CNN, the quantum circuits and all benchmark classifiers are the same code. **What changed is how much data is used and how the results are reported:**

| | Full run (17 Oct notebook) | This notebook |
|---|---|---|
| Data | all eligible subjects: 85 CN + 81 AD, ~22 slices each (3,600 images) | a **small random sample**: `SUBJECTS_PER_CLASS` = 40 CN + 40 AD subjects, `SLICES_PER_SUBJECT` = 10 slices (800 images) |
| Splitting | one 10-fold subject-level CV over all data | the small dataset is **broken into `N_PARTS` = 4 disjoint parts** (10 CN + 10 AD subjects each) |
| Evaluation | fold-wise and pooled results | **each part is its own experiment** (`PART_MODE = "independent"`: 5-fold subject-level CV inside the part) — or, with `PART_MODE = "rotate"`, each part is the test set once |
| Results | Tables 3–11 | for **every classifier in every part**: accuracy (95% CI), balanced accuracy, precision, sensitivity, specificity, NPV, F1, MCC, Cohen's κ, AUC (95% CI), Brier score, log loss, confusion counts, subject-level results, inner-fold mean ± SD, training time; then mean ± SD / min–max over parts, ranks, Friedman test, ROC curves and confusion matrices of every classifier |
| Run time (Kaggle T4) | 2.5–3.5 h | roughly 20–40 min with the defaults; parts can be run in **separate sessions** (`RUN_PARTS`, results saved after every part) |

**The 23 classifiers** (same definitions and fixed hyper-parameters as the full run):
- *Proposed pipeline (fine-tuned VGG16-BN, retrained in every training set):* CNN, VQC (4 qubits), hybrid CNN–VQC, linear ablation on 4 PCA components, linear ablation on 64 PCA components.
- *Benchmark on frozen DenseNet121 + ResNet101 features:* logistic regression, LDA, Gaussian naive Bayes, k-NN, decision tree, linear SVM, RBF SVM, random forest, extra trees, AdaBoost, gradient boosting, XGBoost, LightGBM, MLP, LD-RNN (LSTM), VQC head, hybrid VQC head, soft-voting ensemble.

**Principles kept from the full run**
- No **subject** is ever in more than one of training, validation and test data (checked with assertions in every part and fold), and the parts share no subject.
- The test data are never used for training, early stopping, PCA fitting, scaling or any choice.
- Controls are age-matched (≥ 60 years); every number in the tables and text is computed in this run.

**Read the results as what they are.** A part holds about 20 people, so one subject more or less correct moves a part's subject-level accuracy by ~5 points. That is why every part gets a 95% CI and why the summary uses mean ± SD over parts. A small part is a stress test of each classifier, not a replacement for the full-data result.

Left out to keep it short (they are still in the full notebook): qubit sweep, Grad-CAM/saliency/occlusion maps, shot-noise analysis, leakage demonstration, tabular fusion and the auto-generated manuscript section.

**How to run on Kaggle:** attach the datasets `ninadaithal/imagesoasis` and `jboysen/mri-and-alzheimers`; *Settings → Accelerator: GPU, Internet: On*; check the Configuration cell; *Run All*. Outputs go to `/kaggle/working/qexai_parts_outputs/` (zipped as `qexai_parts_outputs.zip`), including the workbook `tables/results_by_part_and_classifier.xlsx` (one sheet per part and one per classifier).

## 1. Installation
Kaggle images already include PyTorch, scikit-learn, OpenCV, XGBoost and LightGBM. This cell installs only what is missing (Qiskit, the circuit-drawing helper, python-docx, openpyxl, SimpleITK for N4 bias-field correction). It **pins NumPy and SciPy to the versions already installed**, so pip cannot upgrade NumPy underneath the other libraries.

**Internet should be ON** (Kaggle: right-hand panel → *Session options* → *Internet*; needs a phone-verified account) for Qiskit and the ImageNet weights. In this version Qiskit is **optional**: it is only used to cross-check the PyTorch simulator of the quantum circuit and to draw the circuit, so the notebook continues with a warning if it cannot be installed.

Run this notebook on its own, not inside a notebook that has loaded TensorFlow/Keras (TensorFlow reserves the whole GPU).

In [ ]:
import importlib.util, importlib.metadata as md, os, socket, subprocess, sys


def _version(dist):
    try:
        return md.version(dist)
    except md.PackageNotFoundError:
        return None


def _accepts_installed_numpy(dist):
    """Does `dist`'s declared NumPy requirement accept the NumPy installed here?"""
    try:
        from packaging.requirements import Requirement
        for r in md.requires(dist) or []:
            req = Requirement(r)
            if req.name.lower() == "numpy" and req.marker is None:
                return req.specifier.contains(md.version("numpy"), prereleases=True)
    except Exception:
        pass
    return True


def _online(host="pypi.org", port=443, timeout=6):
    try:
        socket.create_connection((host, port), timeout=timeout).close()
        return True
    except OSError:
        return False


OFFLINE_MSG = ("No Internet connection (pip could not reach pypi.org).\n"
               "Kaggle: open the right-hand panel -> Session options / Settings -> switch 'Internet' ON. "
               "The switch is only available after phone verification of your Kaggle account "
               "(kaggle.com -> Settings -> Phone verification). Then restart the session and Run All.\n"
               "The Internet is needed for Qiskit and for the ImageNet weights of the CNNs (downloaded once).")
pins = [f"{d}=={_version(d)}" for d in ("numpy", "scipy") if _version(d)]   # never upgrade these
optional = [spec for module, spec in {"pylatexenc": "pylatexenc", "docx": "python-docx", "openpyxl": "openpyxl",
                                      "SimpleITK": "SimpleITK", "xgboost": "xgboost", "lightgbm": "lightgbm"}.items()
            if importlib.util.find_spec(module) is None]
required = []
if _version("qiskit") is not None and not _accepts_installed_numpy("qiskit"):
    print(f"Installed Qiskit {_version('qiskit')} needs a different NumPy than {_version('numpy')} -> replacing it")
    subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "qiskit"])
if _version("qiskit") is None:
    required.append("qiskit>=1.3,<3")
ONLINE = _online() or bool(os.environ.get("HTTPS_PROXY") or os.environ.get("https_proxy"))   # behind a proxy: let pip try
PIP = [sys.executable, "-m", "pip", "install", "-q", "--retries", "1", "--timeout", "30"]
if required:                      # Qiskit is only used to cross-check the PyTorch circuit simulator: not fatal if missing
    if not ONLINE:
        print("WARNING: offline - Qiskit not installed; the VQC still runs (PyTorch simulator) without the Qiskit "
              "cross-check.\n" + OFFLINE_MSG)
    else:
        print("Installing (required):", required, "| keeping:", pins)
        if subprocess.run([*PIP, *required, *pins]).returncode != 0:
            print("WARNING: pip could not install Qiskit (see the messages above); continuing without the Qiskit cross-check.")
if optional:
    if not ONLINE:
        print("WARNING: offline - optional packages not installed:", optional,
              "\n  -> the circuit is drawn as text, N4 is skipped without SimpleITK, and the .docx/.xlsx exports are "
              "skipped (CSV/Markdown are still written).")
    else:
        print("Installing (optional):", optional)
        for spec in optional:                                 # one by one: a failure here never stops the notebook
            if subprocess.run([*PIP, spec, *pins]).returncode != 0:
                print(f"WARNING: could not install {spec}; continuing without it")
if not ONLINE:
    print("WARNING: no Internet connection. Unless the ImageNet weights are already cached, loading the CNN will fail.\n"
          + OFFLINE_MSG)

# The libraries must still import with the NumPy on disk (an earlier, unpinned install may have upgraded it)
probe = subprocess.run([sys.executable, "-c", "import numpy, scipy, pandas, sklearn, cv2, matplotlib"],
                       capture_output=True, text=True)
if probe.returncode != 0:
    raise SystemExit("The Python environment is inconsistent (usually: NumPy was upgraded by an earlier pip install).\n"
                     "Kaggle: menu Run -> Factory reset, then Run All.\n\n" + probe.stderr[-1500:])
qprobe = subprocess.run([sys.executable, "-c", "import qiskit, numpy"], capture_output=True, text=True,   # separate process
                        env={**os.environ, "GLIBC_TUNABLES": "glibc.rtld.optional_static_tls=4096"})
if qprobe.returncode != 0:
    print("WARNING: Qiskit does not import here; the notebook continues without the Qiskit cross-check.\n"
          + qprobe.stderr[-600:])
loaded = {m for m in ("numpy", "qiskit") if m in sys.modules}
print(f"numpy {_version('numpy')} | scipy {_version('scipy')} | qiskit {_version('qiskit')}")
if loaded:
    print(f"NOTE: {sorted(loaded)} already loaded in this kernel. If anything was installed above, restart the kernel "
          "(Run -> Restart & clear cell outputs) and Run All.")


## 2. Configuration
Edit this cell only. The new settings are in the **SMALL DATASET** and **PARTS** blocks:

| Setting | Default | Meaning |
|---|---|---|
| `SUBJECTS_PER_CLASS` | 40 | subjects drawn at random per class (AD in proportion to its CDR levels); `None` = all eligible subjects |
| `SLICES_PER_SUBJECT` | 10 | highest-entropy slices per subject |
| `N_PARTS` | 4 | number of disjoint, class-balanced parts the small dataset is broken into |
| `PART_MODE` | `"independent"` | `"independent"`: every part is its own experiment (train and test inside the part); `"rotate"`: each part is the test set once, the other parts train |
| `INNER_FOLDS` | 5 | subject-level CV folds inside a part (independent mode) |
| `RUN_PARTS` | `"all"` | run only some parts now, e.g. `[1, 2]`, and the rest later (`[3, 4]`); finished parts are saved and re-loaded (`RESUME`) |
| `RUN_PROPOSED` / `RUN_ML_BENCHMARK` | `True` / `True` | switch the 5 proposed-pipeline classifiers or the 18 benchmark classifiers off |

To continue in a **new Kaggle session**, attach the earlier output as an input and set `RESUME_DIR` to its `results` folder; parts are only reused when all settings and the selected images are identical (checked with a signature).

In [ ]:
import os

# ---- data -------------------------------------------------------------------------------------
DATA_ROOT = os.environ.get("QEXAI_DATA_ROOT", "/kaggle/input")
OUT_DIR = "/kaggle/working/qexai_parts_outputs" if os.path.isdir("/kaggle/working") else os.path.abspath("qexai_parts_outputs")
DATASET_NAME = ("the OASIS-1 cross-sectional MRI data (Marcus et al., 2007) in the 2D-slice version distributed on "
                "Kaggle as “OASIS Alzheimer's Detection”")
DATASET_CITATION = "[cite: Marcus et al., 2007; https://www.kaggle.com/datasets/ninadaithal/imagesoasis]"
AD_MIN_CDR = 0.5          # AD = CDR >= 0.5 (very mild to moderate AD, the OASIS-1 definition); 1.0 = mild + moderate only
CN_MIN_AGE = 60           # age-matching: keep CN subjects aged >= 60 (all OASIS-1 dementia cases are over 60). None = no filter
REQUIRE_DEMOGRAPHICS = True
DEMOGRAPHICS_FILE = None  # explicit path to oasis_cross-sectional.csv/.xlsx, or None (search DATA_ROOT, then download)
DEMOGRAPHICS_URLS = [
    "https://bpb-us-e2.wpmucdn.com/sites.wustl.edu/dist/6/4383/files/2024/04/oasis_cross-sectional-5708aa0a98d82080.xlsx",
    "https://sites.wustl.edu/oasisbrains/files/2024/04/oasis_cross-sectional-5708aa0a98d82080.xlsx",
]

# ---- SMALL DATASET (new in this version) --------------------------------------------------------
# The full run used every eligible subject (85 CN / 81 AD after age matching) and ~22 slices each (3,600 images).
# Here only a small, class-balanced random sample of subjects is used, with fewer slices per subject.
SUBJECTS_PER_CLASS = 40   # subjects drawn per class (CN and AD; AD drawn proportionally to its CDR levels). None = all
SLICES_PER_SUBJECT = 10   # slices per subject (highest-entropy slices of one scan) -> 2 x 40 x 10 = 800 images
SLICE_RANGE = (100, 160)  # axial slice indices to sample from (the Kaggle set contains 100-160)
SLICE_SELECTION = "entropy"   # "entropy" | "even" | "central"
INPUT_MODE = "2.5d"       # "2.5d": slice + neighbours at -/+SLICE_OFFSET as 3 channels | "2d"
SLICE_OFFSET = 2

# ---- PARTS (new in this version) ----------------------------------------------------------------
N_PARTS = 4               # the small dataset is split into N_PARTS disjoint, class-balanced parts (by SUBJECT)
PART_MODE = "independent" # "independent": each part is its own small experiment - every classifier is trained and tested
                          #     inside the part with subject-level INNER_FOLDS-fold CV; nothing is shared between parts
                          # "rotate": each part is the test set once, the other parts are the training data
INNER_FOLDS = 5           # "independent" only: subject-level folds inside a part (every image of the part is tested once)
VAL_FRAC = 0.2            # share of each training set's subjects held out for early stopping / model selection
RUN_PARTS = "all"         # "all", or a list of part numbers, e.g. [1, 2] now and [3, 4] in a later session
RESUME = True             # parts already saved (same settings and images) are loaded instead of re-trained
RESUME_DIR = None         # None = OUT_DIR/results; or the "results" folder of an earlier run attached as a Kaggle input

# ---- pre-processing (Sec. 3.2) ------------------------------------------------------------------
BIAS_CORRECTION = True    # N4 bias-field correction (SimpleITK)
SKULL_STRIP = True        # 2D brain extraction (falls back to the head mask when implausible)
BRAIN_CROP = True         # crop the stack to the brain's bounding box
DENOISE = "gaussian"      # "gaussian" | "median" | "none"
GAUSS_SIGMA = 0.8
CLAHE_CLIP = 2.0
INTENSITY_NORM = "zscore" # "zscore" within the brain mask | "minmax"
PREPROC_PRESET = "qexai"  # "qexai" | "paper" (LD-RNN study: FOV crop, CLAHE 8x8 clip 3.7, [0, 1] scaling)
if PREPROC_PRESET == "paper":
    BIAS_CORRECTION, SKULL_STRIP, BRAIN_CROP, DENOISE, CLAHE_CLIP, INTENSITY_NORM = False, False, False, "none", 3.7, "minmax"

# ---- evaluation ---------------------------------------------------------------------------------
SEED = 42
N_BOOTSTRAP = 500         # subject-level bootstrap resamples for the 95% CIs of each part

# ---- which classifiers ----------------------------------------------------------------------------
RUN_PROPOSED = True       # fine-tuned VGG16-BN CNN -> PCA -> VQC / hybrid CNN-VQC / two linear ablations (5 classifiers)
RUN_ML_BENCHMARK = True   # frozen DenseNet121 + ResNet101 features -> PCA -> 18 benchmark classifiers
ML_FEATURE_SETS = {"dual_cnn": ("densenet121", "resnet101")}   # add "vit": ("vit_b_16",) for Vision-Transformer features
ML_PCA = 1200             # PCA components for the benchmark (LD-RNN study: 1200); capped automatically at (training images - 1)
ML_MODELS = ["logreg", "lda", "gnb", "knn", "dtree", "svm_linear", "svm_rbf", "rf", "extra_trees", "adaboost", "hgb",
             "xgboost", "lightgbm", "mlp", "ldrnn", "vqc_dual", "hybrid_dual", "ensemble"]
ENSEMBLE_MEMBERS = ["logreg", "svm_rbf", "rf", "lightgbm", "ldrnn"]   # soft voting; fixed before seeing any result
ML_TREES = 500
LDRNN_EPOCHS, LDRNN_BATCH, LDRNN_LR, LDRNN_PATIENCE = 70, 16, 1e-3, 10

# ---- CNN (Sec. 3.3) -------------------------------------------------------------------------------
IMG_SIZE = 224
BACKBONE = "vgg16_bn"     # "vgg16_bn" | "vgg16" | "custom"
PRETRAINED = os.environ.get("QEXAI_PRETRAINED", "1") == "1"   # ImageNet weights (Internet ON)
CNN_WIDTHS = (32, 64, 128, 256, 256)
FEATURE_DIM = 4096
CNN_DROPOUT = 0.5
BATCH_SIZE = 16
CNN_LR = 1e-4 if BACKBONE != "custom" else 1e-3
CNN_LR_BACKBONE = 1e-5
CNN_WEIGHT_DECAY = 1e-2
CNN_EPOCHS = 15           # full run: 25 (each part has far fewer images, so each epoch is short)
CNN_MIN_EPOCHS = 4
CNN_PATIENCE = 5
USE_AMP = True
FREEZE_BLOCKS = 3
LABEL_SMOOTHING = 0.1
SELECT_BY = "val_auc"     # "val_auc" | "val_loss"
AUGMENT = "strong"        # "strong" | "basic"
VERBOSE_CNN = False       # True prints every CNN epoch; False prints one summary line per training run

# ---- PCA + quantum part (Sec. 3.4-3.7) ------------------------------------------------------------
N_PCA = 64                # capped automatically at (training images - 1)
N_QUBITS = 4
ENCODING = "zz"           # "zz" | "angle"
FM_REPS = 2
ANSATZ_REPS = 2
ENTANGLEMENT = "linear"
Q_LR = 0.03
Q_EPOCHS = 80
Q_PATIENCE = 10
Q_BATCH = 16
Q_RESTARTS = 3
VERIFY_WITH_QISKIT = True # check the PyTorch circuit simulator against Qiskit's Statevector (skipped if Qiskit is missing)

# ---- smoke test (never report these numbers) ------------------------------------------------------
FAST_DEV_RUN = os.environ.get("QEXAI_FAST_DEV_RUN", "0") == "1"
if FAST_DEV_RUN:
    IMG_SIZE, SUBJECTS_PER_CLASS, SLICES_PER_SUBJECT, N_PARTS, INNER_FOLDS = 64, 12, 4, 2, 3
    CNN_EPOCHS, CNN_MIN_EPOCHS, CNN_PATIENCE, Q_EPOCHS, Q_RESTARTS = 2, 1, 1, 8, 1
    ML_TREES, LDRNN_EPOCHS, N_BOOTSTRAP = 20, 3, 50
    print("FAST_DEV_RUN = True — smoke test only. DO NOT use these outputs in the manuscript.")
IN_CHANNELS = 3 if INPUT_MODE == "2.5d" else 1
assert PART_MODE in ("independent", "rotate"), "PART_MODE must be 'independent' or 'rotate'"


In [ ]:
import contextlib, copy, json, math, os, re, sys, time, random, hashlib, platform, warnings, zipfile, datetime
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image as IPImage
from scipy import stats
import sklearn
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix

os.environ.setdefault("TF_FORCE_GPU_ALLOW_GROWTH", "true")      # in case TensorFlow is imported later
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as T

warnings.filterwarnings("ignore", category=UserWarning)

# ---- environment checks (version mismatch, Qiskit/NumPy compatibility, free GPU memory) ----
import importlib.metadata as _md
if np.__version__ != _md.version("numpy"):
    raise SystemExit(f"This kernel runs NumPy {np.__version__}, but NumPy {_md.version('numpy')} is installed on disk. "
                     "Restart the kernel (Run -> Restart & clear cell outputs) and Run All; if it persists, Run -> Factory reset.")

# ---- Qiskit runs in a SEPARATE Python process ----------------------------------------------------------------------
# Qiskit's compiled Rust library and PyTorch's OpenMP library both need "static TLS" memory, and on recent Kaggle images
# there is not enough for both in one process ("cannot allocate memory in static TLS block"). The kernel therefore never
# imports Qiskit: circuits are built, drawn, verified and sampled by a small worker script started with subprocess.
import subprocess, tempfile
QISKIT_WORKER_SRC = r'''
import json, sys
import numpy as np


def build(n, enc, fm_reps, an_reps, ent):
    from qiskit import QuantumCircuit
    from qiskit.circuit import ParameterVector
    from qiskit.circuit.library import zz_feature_map, real_amplitudes
    if enc == "zz":
        fm = zz_feature_map(n, reps=fm_reps, entanglement=ent)
    else:
        xv = ParameterVector("x", n)
        fm = QuantumCircuit(n, name="AngleEncoding")
        for i in range(n):
            fm.ry(xv[i], i)
    an = real_amplitudes(n, reps=an_reps, entanglement=ent)
    return fm.compose(an), list(fm.parameters), list(an.parameters)


def main(inp, out):
    d = dict(np.load(inp, allow_pickle=False))
    cfg = json.loads(str(d.pop("cfg")))
    import qiskit
    res = {"qiskit_version": np.array(qiskit.__version__)}
    task = cfg["task"]
    if task == "version":
        from qiskit import QuantumCircuit
        from qiskit.quantum_info import Statevector
        qc = QuantumCircuit(1); qc.h(0); Statevector(qc)
    else:
        n = int(cfg["n"])
        qc, xp, tp = build(n, cfg["encoding"], int(cfg["fm_reps"]), int(cfg["ansatz_reps"]), cfg["entanglement"])
        if task == "info":
            res.update(depth=np.array(qc.depth()), cx=np.array(int(qc.count_ops().get("cx", 0))),
                       n_theta=np.array(len(tp)), n_x=np.array(len(xp)), drawn=np.array(0))
            try:
                import matplotlib
                matplotlib.use("Agg")
                fig = qc.draw("mpl", fold=-1, scale=0.7)
                fig.suptitle(cfg["title"], fontsize=10)
                fig.savefig(cfg["png"], dpi=300, bbox_inches="tight")
                fig.savefig(cfg["png"][:-4] + ".pdf", bbox_inches="tight")
                res["drawn"] = np.array(1)
            except Exception as e:                       # pylatexenc missing etc.
                res["draw_error"] = np.array(str(e))
                res["text"] = np.array(str(qc.draw("text", fold=160)))
        elif task == "expect":
            from qiskit.quantum_info import Statevector, SparsePauliOp
            X, th = d["X"].astype(float), d["theta"].astype(float)
            obs = [SparsePauliOp.from_sparse_list([("Z", [k], 1.0)], num_qubits=n) for k in range(n)]
            Z = np.zeros((len(X), n))
            for s_, xs in enumerate(X):
                sv = Statevector(qc.assign_parameters({**dict(zip(xp, xs)), **dict(zip(tp, th))}))
                Z[s_] = [sv.expectation_value(o).real for o in obs]
            res["Z"] = Z
        elif task == "shots":
            from qiskit.primitives import StatevectorSampler
            X, th = d["X"].astype(float), d["theta"].astype(float)
            qc_m = qc.copy(); qc_m.measure_all()
            params = {tuple(xp): X, tuple(tp): np.tile(th, (len(X), 1))}
            zs = []
            for shots in cfg["shots"]:
                for sd in cfg["seeds"]:
                    r = StatevectorSampler(default_shots=int(shots), seed=int(sd)).run([(qc_m, params)]).result()[0]
                    bits = r.data.meas.array                     # (N, shots, bytes), little-endian: qubit k = bit k
                    vals = np.zeros(bits.shape[:-1], dtype=np.int64)
                    for b in range(bits.shape[-1]):
                        vals = (vals << 8) | bits[..., b].astype(np.int64)
                    zs.append(np.stack([1 - 2 * ((vals >> k) & 1).mean(-1) for k in range(n)], 1))
            res["Zs"] = np.stack(zs)
    np.savez(out, **res)


if __name__ == "__main__":
    main(sys.argv[1], sys.argv[2])
'''
QISKIT_WORKER = os.path.join(tempfile.gettempdir(), "qexai_qiskit_worker.py")
with open(QISKIT_WORKER, "w") as fh_:
    fh_.write(QISKIT_WORKER_SRC)


def qiskit_call(task, arrays=None, timeout=7200, **cfg):
    """Run one Qiskit task in a separate process; returns a dict of numpy arrays."""
    cfg = {"encoding": ENCODING, "fm_reps": FM_REPS, "ansatz_reps": ANSATZ_REPS, "entanglement": ENTANGLEMENT, **cfg, "task": task}
    tmp = tempfile.mkdtemp(prefix="qk_")
    inp, out = os.path.join(tmp, "in.npz"), os.path.join(tmp, "out.npz")
    np.savez(inp, cfg=np.array(json.dumps(cfg)), **{k: np.asarray(v) for k, v in (arrays or {}).items()})
    env = dict(os.environ)
    if "optional_static_tls" not in env.get("GLIBC_TUNABLES", ""):   # extra safety margin for the worker process
        env["GLIBC_TUNABLES"] = ":".join(t for t in (env.get("GLIBC_TUNABLES"), "glibc.rtld.optional_static_tls=4096") if t)
    r = subprocess.run([sys.executable, QISKIT_WORKER, inp, out], capture_output=True, text=True, timeout=timeout, env=env)
    if r.returncode != 0 or not os.path.exists(out):
        raise RuntimeError(f"Qiskit worker failed ({task}):\n{r.stderr[-3000:]}")
    with np.load(out, allow_pickle=False) as z:
        return {k: z[k] for k in z.files}


try:
    QISKIT_VERSION, QISKIT_OK = str(qiskit_call("version", timeout=600)["qiskit_version"]), True
except Exception as e:                   # Qiskit is only needed for the cross-check of the simulator and the circuit figure
    QISKIT_VERSION, QISKIT_OK = "not available", False
    print(f"WARNING: Qiskit does not work in this environment ({str(e)[-300:]}).\n"
          "The VQC is still simulated exactly in PyTorch, but without the Qiskit cross-check and circuit drawing.")
if torch.cuda.is_available():
    free_b, total_b = torch.cuda.mem_get_info()
    print(f"GPU memory free: {free_b / 2**30:.1f} of {total_b / 2**30:.1f} GB")
    if free_b < 1.5 * 2**30:
        holders = [m for m in ("tensorflow", "keras", "jax") if m in sys.modules]
        raise SystemExit(
            f"Only {free_b / 2**20:.0f} MB of GPU memory is free: another library or an earlier run is holding it"
            + (f" (loaded here: {holders})" if holders else "") + ".\n"
            "Fix: Run -> Restart & clear cell outputs, then Run All. Run this notebook on its own — do not mix it with "
            "TensorFlow/Keras cells from another notebook in the same session.")
for sub in ("figures", "tables", "results", "cache"):
    os.makedirs(os.path.join(OUT_DIR, sub), exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
Q_DEVICE = torch.device("cpu")   # the small quantum/linear heads train faster on CPU (no GPU kernel-launch overhead)
NUM_WORKERS = 2 if (os.cpu_count() or 1) > 2 else 0


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)
ENV = {"python": platform.python_version(), "torch": torch.__version__, "torchvision": torchvision.__version__,
       "qiskit": QISKIT_VERSION, "scikit-learn": sklearn.__version__, "opencv": cv2.__version__,
       "numpy": np.__version__,
       "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else f"CPU ({os.cpu_count()} cores)"}
print(json.dumps(ENV, indent=1))

# Figure style — categorical slots from a colour-blind-validated palette (first three slots validate all-pairs)
COLORS = {"cnn": "#2a78d6", "vqc": "#eb6834", "hybrid": "#1baf7a", "lin_n": "#8a8984", "lin64": "#52514e"}
CLASS_COLORS = {0: "#2a78d6", 1: "#eb6834"}
PART_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]   # fixed order
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
                     "axes.edgecolor": "#52514e", "axes.labelcolor": "#0b0b0b", "xtick.color": "#52514e",
                     "ytick.color": "#52514e", "legend.frameon": False, "figure.dpi": 100})


def savefig(fig, name):
    path = os.path.join(OUT_DIR, "figures", f"{name}.png")
    fig.savefig(path, dpi=300, bbox_inches="tight")
    fig.savefig(path[:-4] + ".pdf", bbox_inches="tight")
    plt.show()
    plt.close(fig)
    return path


## 3. The small dataset: labels, age-matching, subject sampling and slice selection
Same rules as the full run: one session (MR1) and one acquisition (lowest `mpr`) per subject; AD = CDR ≥ `AD_MIN_CDR`; controls restricted to age ≥ `CN_MIN_AGE` using the OASIS-1 table. **New:** only `SUBJECTS_PER_CLASS` subjects per class are drawn (fixed seed; AD drawn in proportion to CDR 0.5 / 1 / 2), and `SLICES_PER_SUBJECT` highest-entropy slices are taken from each. The sample is drawn before any model is trained, so it cannot favour a classifier.

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png"}
_norm = lambda s: re.sub(r"[^a-z0-9]", "", str(s).lower())
CDR_BY_FOLDER = {"nondemented": 0.0, "verymilddementia": 0.5, "verymilddemented": 0.5, "milddementia": 1.0,
                 "milddemented": 1.0, "moderatedementia": 2.0, "moderatedemented": 2.0}
FNAME_RE = re.compile(r"(OAS1_\d{4})_(MR\d)_mpr-(\d+)_(\d+)", re.IGNORECASE)

rows, unparsed, csv_files = [], [], []
for root, dirs, files in os.walk(DATA_ROOT, followlinks=True):   # Kaggle may mount inputs as symlinks
    dirs.sort()
    for f in sorted(files):
        ext = Path(f).suffix.lower()
        if ext in (".csv", ".xlsx", ".xls"):
            csv_files.append(os.path.join(root, f))
            continue
        if ext not in IMG_EXT:
            continue
        folder = Path(root).name
        cdr = CDR_BY_FOLDER.get(_norm(folder))
        if cdr is None:
            continue
        m = FNAME_RE.search(f)
        if m is None:
            unparsed.append(os.path.join(root, f))
            continue
        p = os.path.join(root, f)
        rows.append(dict(path=p, rel=os.path.relpath(p, DATA_ROOT), folder=folder, cdr_folder=cdr,
                         subject=m.group(1).upper(), session=m.group(2).upper(), mpr=int(m.group(3)), slice=int(m.group(4))))
df_all = pd.DataFrame(rows)
if df_all.empty:
    raise SystemExit(f"No OASIS images found under {DATA_ROOT}. Attach 'ninadaithal/imagesoasis' with 'Add Data'.")
if unparsed:
    print(f"{len(unparsed)} images in class folders do not follow the OASIS filename pattern and were skipped, e.g.:",
          *unparsed[:3], sep="\n  ")
n_raw = len(df_all)
df_all = df_all.drop_duplicates(["subject", "session", "mpr", "slice"]).reset_index(drop=True)   # dataset attached twice etc.
print(f"Images found: {n_raw:,} ({n_raw - len(df_all):,} duplicate copies removed)")
INVENTORY = (df_all.assign(scan=df_all.subject + df_all.session + df_all.mpr.astype(str))
             .groupby(["folder", "cdr_folder"])
             .agg(images=("path", "size"), subjects=("subject", "nunique"), scans=("scan", "nunique"),
                  first_slice=("slice", "min"), last_slice=("slice", "max")).reset_index().sort_values("cdr_folder"))
display(INVENTORY)
INVENTORY.to_csv(os.path.join(OUT_DIR, "tables", "data_inventory.csv"), index=False)
_probe = cv2.imread(df_all.path.iloc[0], cv2.IMREAD_UNCHANGED)
print("Example file:", df_all.rel.iloc[0], "| stored size (H, W[, C]):", None if _probe is None else _probe.shape)

# a subject must sit in exactly one CDR folder
n_folders = df_all.groupby("subject").folder.nunique()
if (n_folders > 1).any():
    bad = n_folders[n_folders > 1].index
    print(f"Dropping {len(bad)} subjects that appear in more than one CDR folder:", list(bad)[:10])
    df_all = df_all[~df_all.subject.isin(bad)].reset_index(drop=True)

# ---- OASIS-1 demographics (oasis_cross-sectional.csv / .xlsx) ----
def read_oasis_demographics(path, log=None):
    """Return the OASIS-1 table indexed by subject (MR1 rows), or None if the file is not that table."""
    log = log if log is not None else []
    try:
        if str(path).lower().endswith((".xlsx", ".xls")):
            raw = pd.read_excel(path, header=None)
            hdr = next((i for i in range(min(15, len(raw))) if "ID" in [str(v).strip() for v in raw.iloc[i]]), None)
            if hdr is None:
                log.append(f"{path}: no 'ID' header row"); return None
            d = pd.read_excel(path, header=hdr)
        else:
            d = pd.read_csv(path, encoding="utf-8-sig", sep=None, engine="python")
    except Exception as e:
        log.append(f"{path}: could not be read ({type(e).__name__}: {e})"); return None
    d.columns = [str(c).replace("\ufeff", "").strip() for c in d.columns]
    if not {"ID", "Age", "CDR"} <= set(d.columns):
        log.append(f"{path}: columns {list(d.columns)[:8]} (needs ID, Age, CDR)"); return None
    ids = d.ID.astype(str).str.strip().str.upper()
    if not ids.str.startswith("OAS1_").any():
        log.append(f"{path}: IDs are not OASIS-1 (e.g. {ids.iloc[0] if len(ids) else 'empty'})"); return None
    d = d[ids.str.endswith("_MR1")].copy()
    d["subject"] = d.ID.astype(str).str.strip().str[:9].str.upper()
    for c in ("Age", "MMSE", "CDR", "Educ", "SES", "eTIV", "nWBV", "ASF"):
        if c in d:
            d[c] = pd.to_numeric(d[c], errors="coerce")
    if "M/F" in d:
        d["M/F"] = d["M/F"].astype(str).str.strip().str.upper()
    return d.drop_duplicates("subject").set_index("subject")


def download(url, dst):
    """Browser-like download; returns (ok, message). Accepts only an Excel (zip) or CSV payload."""
    headers = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124 Safari/537.36",
               "Accept": "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet,text/csv,*/*"}
    try:
        try:
            import requests
            r = requests.get(url, headers=headers, timeout=60, allow_redirects=True)
            status, data = r.status_code, r.content
        except ImportError:
            import urllib.request
            with urllib.request.urlopen(urllib.request.Request(url, headers=headers), timeout=60) as r:
                status, data = r.status, r.read()
    except Exception as e:
        return False, f"{type(e).__name__}: {str(e)[:160]}"
    if status != 200:
        return False, f"HTTP {status}"
    if not (data[:2] == b"PK" or data[:3] == b"ID," or data[:6] == b"\xef\xbb\xbfID,"):
        return False, f"unexpected content ({len(data)} bytes, starts {data[:40]!r})"
    with open(dst, "wb") as fh:
        fh.write(data)
    return True, f"{len(data):,} bytes"


DEMO, DEMO_PATH, demo_log = None, None, []
candidates = ([DEMOGRAPHICS_FILE] if DEMOGRAPHICS_FILE else []) + sorted(csv_files)
for cp in candidates:                               # 1) + 2) files you attached / uploaded
    DEMO = read_oasis_demographics(cp, demo_log)
    if DEMO is not None:
        DEMO_PATH = cp
        break
if DEMO is None:                                    # 3) official OASIS-1 spreadsheet (Internet ON)
    for url in DEMOGRAPHICS_URLS or []:
        dst = os.path.join(OUT_DIR, "oasis_cross-sectional" + (".xlsx" if url.lower().endswith(".xlsx") else ".csv"))
        ok_dl, msg = download(url, dst)
        demo_log.append(f"download {url[:70]}... -> {msg}")
        if ok_dl:
            DEMO = read_oasis_demographics(dst, demo_log)
            if DEMO is not None:
                DEMO_PATH = url + " (downloaded)"
                break
if DEMO is None:
    print("Demographics search log:", *(demo_log or ["no .csv/.xlsx files under DATA_ROOT and no download URL"]), sep="\n  ")
if DEMO is None and REQUIRE_DEMOGRAPHICS:
    raise SystemExit(
        "OASIS-1 demographics (age/sex/MMSE/CDR) not found — see the search log printed above. Fix with ONE of these:\n"
        "  (a) Kaggle: right panel -> '+ Add Input' -> Datasets -> search 'MRI and Alzheimers' (owner: jboysen) -> Add;\n"
        "  (b) download https://sites.wustl.edu/oasisbrains/files/2024/04/oasis_cross-sectional-5708aa0a98d82080.xlsx\n"
        "      in your browser, then Kaggle: '+ Add Input' -> 'Upload' -> New dataset -> drop the .xlsx -> Create,\n"
        "      and set DEMOGRAPHICS_FILE to its path (shown under Input in the right panel), or leave it None;\n"
        "  (c) last resort: set REQUIRE_DEMOGRAPHICS = False (controls then include adults aged 18-59: age confound).\n"
        "Then re-run the Configuration cell and this cell.")
print("Demographics:", DEMO_PATH or "NOT AVAILABLE — controls cannot be age-matched",
      "" if DEMO is None else f"({len(DEMO)} OASIS-1 subjects)")

SUBJ_TAB = df_all.groupby("subject").agg(folder=("folder", "first"), cdr=("cdr_folder", "first")).reset_index()
if DEMO is not None:
    keep_cols = [c for c in ("M/F", "Age", "Educ", "SES", "MMSE", "CDR", "eTIV", "nWBV", "ASF") if c in DEMO.columns]
    SUBJ_TAB = SUBJ_TAB.join(DEMO[keep_cols], on="subject")
    mism = SUBJ_TAB[SUBJ_TAB.CDR.notna() & (SUBJ_TAB.CDR != SUBJ_TAB.cdr)]
    if len(mism):
        print(f"Dropping {len(mism)} subjects whose folder CDR differs from the CSV CDR:", list(mism.subject)[:10])
        SUBJ_TAB = SUBJ_TAB.drop(mism.index)
    print(f"Subjects with demographics: {int(SUBJ_TAB.Age.notna().sum())} of {len(SUBJ_TAB)}")
SUBJ_TAB["label"] = np.where(SUBJ_TAB.cdr >= AD_MIN_CDR, 1, np.where(SUBJ_TAB.cdr == 0, 0, -1))
SUBJ_TAB["reason"] = np.where(SUBJ_TAB.label == -1, f"CDR between 0 and {AD_MIN_CDR:g}", "")
AGE_MATCHED = DEMO is not None and CN_MIN_AGE is not None
if AGE_MATCHED:
    young = (SUBJ_TAB.label == 0) & ~(SUBJ_TAB.Age >= CN_MIN_AGE)
    SUBJ_TAB.loc[young, ["label", "reason"]] = [-1, f"CN younger than {CN_MIN_AGE} (or age missing)"]
else:
    print("WARNING: controls are NOT age-matched; the generated text states this as a limitation.")
display(SUBJ_TAB.assign(group=SUBJ_TAB.label.map({0: "CN", 1: "AD", -1: "excluded"}))
        .groupby(["folder", "group", "reason"]).size().rename("subjects").reset_index())

# ---- slice selection: one session + one acquisition per subject; slices chosen without looking at labels ----
rng = np.random.default_rng(SEED)


def slice_entropy(path):
    """Shannon entropy (bits) of the grey-level histogram of the whole slice (Hon & Khan 2017)."""
    im = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if im is None:
        return -1.0
    p_ = np.bincount(im.ravel(), minlength=256).astype(float)
    p_ = p_[p_ > 0] / p_.sum()
    return float(-(p_ * np.log2(p_)).sum())


def choose_slices(g, k):
    """k slices of one scan (g sorted by slice number)."""
    if k >= len(g):
        return g
    if SLICE_SELECTION == "entropy":
        ent = np.array([slice_entropy(p_) for p_ in g.path])
        return g.iloc[np.sort(np.argsort(-ent, kind="stable")[:k])]
    if SLICE_SELECTION == "central":
        mid = (g.slice.min() + g.slice.max()) / 2
        return g.iloc[np.sort(np.argsort(np.abs(g.slice.values - mid), kind="stable")[:k])]
    return g.iloc[np.unique(np.round(np.linspace(0, len(g) - 1, k)).astype(int))]


# ---- SMALL DATASET: a random, class-balanced sample of SUBJECTS_PER_CLASS subjects per class ----
# Subjects are drawn with a fixed seed; the label is used only to sample equal numbers per class (AD proportionally
# to its CDR levels). No result is looked at, so the sample cannot favour any classifier.
def stratified_take(table, n, strata_col, rng_):
    """n subjects from `table`, allocated to the levels of `strata_col` in proportion (largest remainder)."""
    levels = table[strata_col].value_counts().sort_index()
    raw = levels / levels.sum() * n
    alloc = np.floor(raw).astype(int)
    for lv in (raw - alloc).sort_values(ascending=False).index[:n - alloc.sum()]:
        alloc.loc[lv] += 1
    out = []
    for lv, k_lv in alloc.items():
        cand = table.loc[table[strata_col] == lv, "subject"].values
        out += list(rng_.choice(cand, min(k_lv, len(cand)), replace=False))
    return sorted(out)


eligible = {lab: SUBJ_TAB[SUBJ_TAB.label == lab] for lab in (0, 1)}
n_take = min(len(eligible[0]), len(eligible[1]))
if SUBJECTS_PER_CLASS is not None:
    n_take = min(n_take, int(SUBJECTS_PER_CLASS))
SMALL_SUBJECTS = {lab: stratified_take(eligible[lab], n_take, "cdr", rng) for lab in (0, 1)}
print(f"Small dataset: {n_take} CN + {n_take} AD subjects drawn from {len(eligible[0])} CN / {len(eligible[1])} AD "
      f"eligible subjects (seed {SEED})")

pick = []
for lab in (0, 1):
    for sbj in SMALL_SUBJECTS[lab]:
        g = df_all[df_all.subject == sbj]
        ses = "MR1" if (g.session == "MR1").any() else sorted(g.session.unique())[0]
        g = g[g.session == ses]
        g = g[(g.mpr == g.mpr.min()) & g.slice.between(*SLICE_RANGE)].sort_values("slice")
        k = min(int(SLICES_PER_SUBJECT), len(g))
        if k:
            pick.append(choose_slices(g, k))
df = pd.concat(pick).reset_index(drop=True)
df["label"] = df.subject.map(SUBJ_TAB.set_index("subject").label).astype(int)
while (df.label == 0).sum() != (df.label == 1).sum():          # exact class balance: trim the larger class
    big = int((df.label == 1).sum() > (df.label == 0).sum())
    counts = df[df.label == big].subject.value_counts()
    df = df.drop(df[(df.subject == counts.index[0])].index[-1])
df = df.reset_index(drop=True)
if INPUT_MODE == "2.5d":   # neighbouring slices of the same scan (nearest available slice if one is missing)
    scan_slices = {key: g.set_index("slice").path.to_dict()
                   for key, g in df_all.groupby(["subject", "session", "mpr"])}
    def _nearest(d, target, centre):
        cand = sorted(d, key=lambda s_: (abs(s_ - target), s_ == centre))
        return d[cand[0]]
    df["path_lo"] = [_nearest(scan_slices[(r.subject, r.session, r.mpr)], r.slice - SLICE_OFFSET, r.slice) for r in df.itertuples()]
    df["path_hi"] = [_nearest(scan_slices[(r.subject, r.session, r.mpr)], r.slice + SLICE_OFFSET, r.slice) for r in df.itertuples()]
print(f"Selected {len(df)} images: CN {int((df.label == 0).sum())} from {df[df.label == 0].subject.nunique()} subjects, "
      f"AD {int((df.label == 1).sum())} from {df[df.label == 1].subject.nunique()} subjects; "
      f"slices per subject {df.groupby('subject').size().min()}–{df.groupby('subject').size().max()} "
      f"({SLICE_SELECTION} selection; slice numbers used {df.slice.min()}–{df.slice.max()}, median {int(df.slice.median())})")
display(df.drop_duplicates("subject").assign(group=lambda d: d.label.map({0: "CN", 1: "AD"}))
        .groupby(["group", "folder"]).size().rename("subjects in the small dataset").reset_index())


In [ ]:
# ---- groups for splitting: subject, merged only if two subjects share a pixel-identical image (copied file) ----
# (A perceptual hash is NOT used: atlas-registered slices at the same position look alike across different people.)
hashes, ok = [], []
for p in df.path:
    im = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    ok.append(im is not None and im.size > 0)
    hashes.append(hashlib.md5(im.tobytes()).hexdigest() if ok[-1] else None)
df["dhash"] = hashes
print("Unreadable files skipped:", int((~np.array(ok)).sum()))
df = df[np.array(ok)].reset_index(drop=True)
parent = {}
def find(a):
    parent.setdefault(a, a)
    while parent[a] != a:
        parent[a] = parent[parent[a]]
        a = parent[a]
    return a
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra
for i, r in df.iterrows():
    union(f"subj:{r.subject}", f"hash:{r.dhash}")
df["group"] = pd.factorize(pd.Series([find(f"subj:{s_}") for s_ in df.subject]))[0]
merged = df.subject.nunique() - df.group.nunique()
print(f"Groups: {df.group.nunique()} ({merged} subjects merged because they shared a pixel-identical image)")
HAS_SUBJECT_IDS, GROUP_UNIT = True, "subject"

# ---- Table 1: demographics of the selected subjects ----
def _msd(v, nd=1):
    v = pd.to_numeric(v, errors="coerce").dropna()
    return f"{v.mean():.{nd}f} ± {v.std(ddof=1):.{nd}f}" if len(v) else "n/a"


sel_subj = SUBJ_TAB[SUBJ_TAB.subject.isin(df.subject.unique())].set_index("subject")
rows1 = []
for name, lab in (("CN", 0), ("AD", 1)):
    t = sel_subj[sel_subj.label == lab]
    r = dict(Group=name, Subjects=len(t), Images=int((df.label == lab).sum()),
             CDR=", ".join(f"{c:g}: {n}" for c, n in t.cdr.value_counts().sort_index().items()))
    if DEMO is not None:
        r.update({"Age (years)": f"{_msd(t.Age)} ({int(t.Age.min())}–{int(t.Age.max())})",
                  "Sex (F/M)": f"{int((t['M/F'] == 'F').sum())}/{int((t['M/F'] == 'M').sum())}" if "M/F" in t else "n/a",
                  "MMSE": _msd(t.MMSE) if "MMSE" in t else "n/a"})
    rows1.append(r)
TABLE1 = pd.DataFrame(rows1)
DEMO_TESTS = {}
if DEMO is not None:
    cn_s, ad_s = sel_subj[sel_subj.label == 0], sel_subj[sel_subj.label == 1]
    DEMO_TESTS["age_p"] = float(stats.ttest_ind(cn_s.Age.dropna(), ad_s.Age.dropna(), equal_var=False).pvalue)
    DEMO_TESTS.update(age_cn=_msd(cn_s.Age), age_ad=_msd(ad_s.Age),
                      mmse_cn=_msd(cn_s.MMSE) if "MMSE" in sel_subj else "n/a", mmse_ad=_msd(ad_s.MMSE) if "MMSE" in sel_subj else "n/a")
    if "M/F" in sel_subj:
        DEMO_TESTS["sex_p"] = float(stats.chi2_contingency(pd.crosstab(sel_subj.label, sel_subj["M/F"]))[1])
    if "MMSE" in sel_subj:
        DEMO_TESTS["mmse_p"] = float(stats.ttest_ind(cn_s.MMSE.dropna(), ad_s.MMSE.dropna(), equal_var=False).pvalue)
    TABLE1 = pd.concat([TABLE1, pd.DataFrame([dict(Group="p (CN vs AD)", Subjects="", Images="", CDR="",
                         **{"Age (years)": f"{DEMO_TESTS['age_p']:.3f}", "Sex (F/M)": f"{DEMO_TESTS.get('sex_p', np.nan):.3f}",
                            "MMSE": f"{DEMO_TESTS.get('mmse_p', np.nan):.3g}"})])], ignore_index=True)
TABLE1.to_csv(os.path.join(OUT_DIR, "tables", "table1_demographics.csv"), index=False)
display(TABLE1)


## 4. MRI pre-processing (Sec. 3.2)
The Kaggle images are 2D JPEG slices of the raw OASIS-1 T1 scans: they **still contain the skull, scalp and eyes**, and they are not bias-field corrected. Each stack (the selected slice and, in 2.5D mode, its two neighbours) goes through these steps, all controlled in the Configuration cell:

| Step | Setting | Why |
|---|---|---|
| 1. Crop to the head, pad to a square, resize to `IMG_SIZE` | always | the Kaggle images are 248 × 496 with a uniform background |
| 2. N4 bias-field correction | `BIAS_CORRECTION` | removes the smooth intensity inhomogeneity of raw MRI; the field is estimated on the centre slice and applied to the whole stack |
| 3. Skull stripping (2D brain extraction) | `SKULL_STRIP` | ray casting from the head centre: on each of 180 rays, the scalp → dark skull/CSF band → brain-surface transition is located, the edge is smoothed across rays and the contour filled. Keeps the cortex; slices where too few rays give an edge fall back to the head mask (rate reported, Figure S5) |
| 4. Crop to the brain | `BRAIN_CROP` | the brain fills the network input instead of ~60% of it |
| 5. Denoising | `DENOISE` (Gaussian, σ = `GAUSS_SIGMA`) | manuscript Eq. 3 |
| 6. CLAHE | clip `CLAHE_CLIP`, 8 × 8 tiles | local tissue contrast |
| 7. Intensity normalisation | `INTENSITY_NORM` (z-score within the brain mask, clipped to ±3 SD) | manuscript Eq. 4; makes intensities comparable across scans |

Every step uses only the image itself (no labels, no other subjects), so it cannot leak information between folds. The brain mask is also used to check where Grad-CAM attention falls.

In [ ]:
CLAHE = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=(8, 8))
cv2.setNumThreads(max(1, os.cpu_count() or 1))


def content_box(gray, thr=12, margin=0.04):
    """Bounding box of the head (pixels differing from the border/background value), with a small margin."""
    border = np.concatenate([gray[0], gray[-1], gray[:, 0], gray[:, -1]])
    bg = int(np.median(border))
    ys, xs = np.where(np.abs(gray.astype(np.int16) - bg) > thr)
    if len(ys) <= 50:
        return None
    y0, y1, x0, x1 = ys.min(), ys.max(), xs.min(), xs.max()
    my, mx = int((y1 - y0) * margin) + 2, int((x1 - x0) * margin) + 2
    return max(0, y0 - my), y1 + my + 1, max(0, x0 - mx), x1 + mx + 1


def crop_pad_square(gray, box="auto"):
    """Crop to the content bounding box (or a given box, so neighbouring slices stay aligned), pad to a square."""
    border = np.concatenate([gray[0], gray[-1], gray[:, 0], gray[:, -1]])
    bg = int(np.median(border))
    box = content_box(gray) if isinstance(box, str) else box
    if box is not None:
        gray = gray[box[0]:box[1], box[2]:box[3]]
    h, w = gray.shape
    side = max(h, w)
    out = np.full((side, side), bg, np.uint8)
    out[(side - h) // 2:(side - h) // 2 + h, (side - w) // 2:(side - w) // 2 + w] = gray
    return out


def disk(r):
    return cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * r + 1, 2 * r + 1))


def fill_holes(m):
    """Background components that do not touch the image border belong to the object (e.g. ventricles)."""
    m = m.astype(np.uint8)
    _, bg_lab = cv2.connectedComponents((m == 0).astype(np.uint8))
    border = np.unique(np.concatenate([bg_lab[0], bg_lab[-1], bg_lab[:, 0], bg_lab[:, -1]]))
    return (m > 0) | ((m == 0) & ~np.isin(bg_lab, border))


def brain_mask(img):
    """Head (foreground) mask: Otsu threshold, closing, largest component, hole filling."""
    _, m = cv2.threshold(cv2.GaussianBlur(img, (5, 5), 0), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((7, 7), np.uint8))
    n_lab, lab, st, _ = cv2.connectedComponentsWithStats(m)
    if n_lab > 1:
        m = np.where(lab == 1 + np.argmax(st[1:, cv2.CC_STAT_AREA]), 255, 0).astype(np.uint8)
    return fill_holes(m)


def _circ_nanmedian(a, w):
    ext = np.concatenate([a[-w:], a, a[:w]])
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)          # all-NaN windows stay NaN
        return np.array([np.nanmedian(ext[i:i + 2 * w + 1]) for i in range(len(a))])


def skull_strip(img, n_rays=180, max_depth=0.25, min_rays=0.35, min_ratio=0.45, max_ratio=0.95):
    """2D brain extraction by ray casting (keeps the cortex). Along rays from the head centre, find the transition
    scalp -> dark skull/CSF band -> brain tissue; median-filter the edge depth across neighbouring rays (outliers from
    orbits/sinuses removed), fill the contour. Returns (mask, ok); if ok is False the head mask is returned."""
    head = brain_mask(img)
    if head.sum() < 0.03 * img.size:
        return head, False
    sm = cv2.GaussianBlur(img, (3, 3), 0).astype(np.float32)
    v = sm[head].astype(np.uint8)
    t_hi, _ = cv2.threshold(v.reshape(-1, 1), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    low = v[v < t_hi]
    if low.size < 50:
        return head, False
    t_lo, _ = cv2.threshold(low.reshape(-1, 1), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)   # dark (bone, CSF) vs tissue
    ys, xs = np.nonzero(head)
    cy, cx = ys.mean(), xs.mean()
    H, W = img.shape
    ang = np.linspace(0, 2 * np.pi, n_rays, endpoint=False)
    step = 0.5
    rr = np.arange(0, np.hypot(H, W), step)
    r_head, depth = np.full(n_rays, np.nan), np.full(n_rays, np.nan)
    for i, a in enumerate(ang):
        yy, xx = cy + rr * np.sin(a), cx + rr * np.cos(a)
        inside = (yy >= 0) & (yy < H - 1) & (xx >= 0) & (xx < W - 1)
        yy, xx = yy[inside], xx[inside]
        hv = head[yy.round().astype(int), xx.round().astype(int)]
        if not hv.any():
            continue
        last = int(np.where(hv)[0].max())
        r_head[i] = rr[last]
        prof = cv2.remap(sm, xx[:last + 1].astype(np.float32).reshape(1, -1), yy[:last + 1].astype(np.float32).reshape(1, -1),
                         cv2.INTER_LINEAR).ravel()[::-1][:int(max_depth * rr[last] / step)]   # head surface -> inwards
        bright = np.where(prof > t_lo)[0]                         # scalp
        if len(bright) == 0:
            continue
        dark = np.where(prof[bright[0]:] <= t_lo)[0]              # skull / CSF band
        if len(dark) == 0:
            continue
        j1 = bright[0] + dark[0]
        back = np.where(prof[j1:] > t_lo)[0]                      # brain surface
        if len(back):
            depth[i] = (j1 + back[0]) * step
    if np.isfinite(depth).mean() < min_rays:
        return head, False
    rel = depth / r_head
    depth[np.abs(rel - np.nanmedian(rel)) > 0.08] = np.nan        # outliers (orbits, sinuses, deep sulci)
    d = _circ_nanmedian(depth, 6)
    ok_d = np.isfinite(d) & np.isfinite(r_head)
    if ok_d.mean() < min_rays:
        return head, False
    idx = np.arange(n_rays)
    d = np.interp(idx, idx[ok_d], d[ok_d], period=n_rays)
    rh = np.interp(idx, idx[np.isfinite(r_head)], r_head[np.isfinite(r_head)], period=n_rays)
    rb = np.clip(rh - d, 0, None)
    pts = np.stack([cx + rb * np.cos(ang), cy + rb * np.sin(ang)], 1).round().astype(np.int32)
    mask = np.zeros(img.shape, np.uint8)
    cv2.fillPoly(mask, [pts], 1)
    mask = (mask > 0) & head
    ok = min_ratio <= mask.sum() / max(1, head.sum()) <= max_ratio
    return (mask if ok else head), bool(ok)


N4_ON = BIAS_CORRECTION
if BIAS_CORRECTION:
    try:
        import SimpleITK as sitk
    except ImportError:
        print("SimpleITK is not installed -> N4 bias-field correction is skipped (re-run the installation cell)")
        N4_ON = False


def n4_log_field(img, mask, shrink=None, iters=(20, 20, 10), max_ratio=2.5):
    """Log bias field of one slice (N4 at ~64 px resolution, returned at full resolution). Returns None if N4 fails or
    the field is implausible (brightest/darkest gain inside the head > max_ratio), so the slice stays uncorrected."""
    shrink = shrink or max(1, int(round(img.shape[0] / 64)))
    try:
        im = sitk.GetImageFromArray(img.astype(np.float32) + 1.0)
        mk = sitk.GetImageFromArray(mask.astype(np.uint8))
        f = sitk.N4BiasFieldCorrectionImageFilter()
        f.SetMaximumNumberOfIterations(list(iters))
        f.Execute(sitk.Shrink(im, [shrink] * 2), sitk.Shrink(mk, [shrink] * 2))
        lf = sitk.GetArrayFromImage(f.GetLogBiasFieldAsImage(im)).astype(np.float32)
    except Exception:
        return None
    if not np.isfinite(lf).all() or (mask.any() and float(np.ptp(lf[mask])) > np.log(max_ratio)):
        return None
    return lf


def _resize(a, interp=cv2.INTER_AREA):
    return cv2.resize(a, (IMG_SIZE, IMG_SIZE), interpolation=interp)


def apply_n4(ims, c=0):
    """Estimate the bias field on ims[c] and divide every image of the stack by it (rescaled to uint8)."""
    head = brain_mask(ims[c])
    fld = n4_log_field(ims[c], head)
    if fld is None:
        return ims
    cor = [im.astype(np.float32) / np.exp(fld) for im in ims]
    hi = max(float(np.percentile(cor[c][head], 99.5)) if head.any() else float(cor[c].max()), 1e-6)
    return [np.clip(x_ / hi * 255, 0, 255).astype(np.uint8) for x_ in cor]


def preprocess_stack(paths, c=0, keep_steps=False):
    """paths: slice files of one stack (centre at index c). Returns (list of uint8 images, centre mask, mask ok, steps)."""
    raws = [cv2.imread(p_, cv2.IMREAD_GRAYSCALE) for p_ in paths]
    box = content_box(raws[c])                                    # one crop for the whole stack
    ims = [_resize(crop_pad_square(r_, box)) for r_ in raws]
    steps = {"Cropped, padded, resized": ims[c].copy()} if keep_steps else None
    if N4_ON:                                                     # 2) N4 bias field from the centre slice
        ims = apply_n4(ims, c)
        if keep_steps:
            steps["N4 bias-field corrected"] = ims[c].copy()
    if SKULL_STRIP:                                               # 3) brain extraction
        m_c, ok = skull_strip(ims[c])
        masks = []
        for i_, im in enumerate(ims):
            if i_ == c or paths[i_] == paths[c]:
                masks.append(m_c)
            else:
                m_i, ok_i = skull_strip(im)
                masks.append(m_i if (ok_i and ok) else m_c)
        ims = [np.where(m_, im, 0).astype(np.uint8) for m_, im in zip(masks, ims)]
        if keep_steps:
            steps["Skull stripped" + ("" if ok else " (fallback: head mask)")] = ims[c].copy()
    else:
        m_c, ok = brain_mask(ims[c]), True
        masks = [m_c] * len(ims)
    if SKULL_STRIP and BRAIN_CROP and m_c.any():                  # 4) crop to the brain (shared box)
        ys, xs = np.nonzero(m_c)
        cy, cx = (ys.min() + ys.max()) / 2, (xs.min() + xs.max()) / 2
        half = max(ys.max() - ys.min(), xs.max() - xs.min()) / 2 * 1.06 + 2
        y0, x0 = int(max(0, cy - half)), int(max(0, cx - half))
        y1, x1 = int(min(IMG_SIZE, cy + half + 1)), int(min(IMG_SIZE, cx + half + 1))
        ims = [_resize(crop_pad_square(im, (y0, y1, x0, x1)), cv2.INTER_LINEAR) for im in ims]
        masks = [_resize(crop_pad_square(m_.astype(np.uint8), (y0, y1, x0, x1)), cv2.INTER_NEAREST) > 0 for m_ in masks]
        m_c = masks[c]
        if keep_steps:
            steps["Cropped to the brain"] = ims[c].copy()
    out = []
    for i_, (im, m_) in enumerate(zip(ims, masks)):
        if DENOISE == "gaussian":                                 # 5) denoising
            im = cv2.GaussianBlur(im, (0, 0), GAUSS_SIGMA)
        elif DENOISE == "median":
            im = cv2.medianBlur(im, 3)
        if keep_steps and i_ == c and DENOISE != "none":
            steps["Gaussian filter (σ = %g)" % GAUSS_SIGMA if DENOISE == "gaussian" else "Median filter (3×3)"] = im.copy()
        im = CLAHE.apply(im)                                      # 6) CLAHE
        if SKULL_STRIP:
            im = np.where(m_, im, 0).astype(np.uint8)
        if keep_steps and i_ == c:
            steps["CLAHE"] = im.copy()
        if INTENSITY_NORM == "zscore" and m_.sum() > 20:          # 7) intensity normalisation
            v_ = im[m_].astype(np.float32)
            z = (im.astype(np.float32) - v_.mean()) / max(float(v_.std()), 1e-6)
            im = np.clip((z + 3) / 6 * 255, 0, 255).astype(np.uint8)
            if SKULL_STRIP:
                im = np.where(m_, im, 0).astype(np.uint8)
        else:
            im = cv2.normalize(im, None, 0, 255, cv2.NORM_MINMAX)
        if keep_steps and i_ == c:
            steps["z-score normalised" if INTENSITY_NORM == "zscore" else "Min–max normalised"] = im.copy()
        out.append(im)
    return out, m_c, ok, steps


X = np.zeros((len(df), IMG_SIZE, IMG_SIZE), np.uint8)          # centre slice (display, masks, 2D input)
X3 = np.zeros((len(df), 3, IMG_SIZE, IMG_SIZE), np.uint8) if INPUT_MODE == "2.5d" else None
MASK = np.zeros((len(df), IMG_SIZE, IMG_SIZE), bool)            # brain (or head) mask of the centre slice
STRIP_OK = np.ones(len(df), bool)
t_pre = time.time()
for i, r in enumerate(df.itertuples()):
    paths, c = ([r.path_lo, r.path, r.path_hi], 1) if X3 is not None else ([r.path], 0)
    chans, MASK[i], STRIP_OK[i], _ = preprocess_stack(paths, c)
    X[i] = chans[c]
    if X3 is not None:
        X3[i] = np.stack(chans)
    if (i + 1) % 200 == 0:
        el = time.time() - t_pre
        print(f"  pre-processed {i + 1}/{len(df)} stacks ({el / 60:.1f} min, ~{el / (i + 1) * (len(df) - i - 1) / 60:.1f} min left)")
y = df.label.values.astype(np.int64)
groups = df.group.values
PREPROC = dict(n4=N4_ON, skull_strip=SKULL_STRIP, strip_ok_pct=100 * float(STRIP_OK.mean()),
               minutes=(time.time() - t_pre) / 60)
print("Image tensor:", (X3 if X3 is not None else X).shape, "| input mode:", INPUT_MODE,
      f"| N4: {'on' if N4_ON else 'off'} | skull stripping succeeded on {PREPROC['strip_ok_pct']:.1f}% of slices"
      if SKULL_STRIP else "", f"| {PREPROC['minutes']:.1f} min")
PREPROC_TXT = "; ".join(t_ for t_ in [
    "crop to the head, pad to a square, resize",
    "N4 bias-field correction (field from the centre slice)" if N4_ON else ("N4 skipped (SimpleITK unavailable)" if BIAS_CORRECTION else ""),
    f"2D skull stripping (brain mask accepted on {PREPROC['strip_ok_pct']:.1f}% of slices, head mask otherwise)" if SKULL_STRIP else "skull not removed",
    "crop to the brain" if SKULL_STRIP and BRAIN_CROP else "",
    {"gaussian": f"Gaussian filter (σ = {GAUSS_SIGMA:g} px)", "median": "median filter 3×3", "none": ""}[DENOISE],
    f"CLAHE ({CLAHE_CLIP:g}, 8×8)",
    ("z-score within the brain mask (clipped to ±3 SD)" if SKULL_STRIP else "z-score within the head mask (clipped to ±3 SD)")
    if INTENSITY_NORM == "zscore" else "min–max normalisation"] if t_)
if SKULL_STRIP and PREPROC["strip_ok_pct"] < 70:
    print("WARNING: skull stripping fell back to the head mask on many slices. Check Figure S5; if the masks are poor, "
          "set SKULL_STRIP = False.")

# Figure 1 / Figure 4: samples and the pre-processing pipeline
demo_i = int(rng.choice(np.where(y == 1)[0]))
demo_paths, demo_c = (([df.path_lo[demo_i], df.path[demo_i], df.path_hi[demo_i]], 1) if X3 is not None else ([df.path[demo_i]], 0))
_, _, _, demo_steps = preprocess_stack(demo_paths, demo_c, keep_steps=True)
raw = cv2.imread(df.path[demo_i], cv2.IMREAD_GRAYSCALE)
panels = [(f"Original ({raw.shape[1]}×{raw.shape[0]})", raw)] + list(demo_steps.items())
ncol = max(6, len(panels))
fig, axes = plt.subplots(3, ncol, figsize=(2.15 * ncol, 7.4))
for row, lab in enumerate((0, 1)):
    picks = rng.choice(np.where(y == lab)[0], ncol, replace=False)
    for ax, i in zip(axes[row], picks):
        ax.imshow(X[i], cmap="gray"); ax.axis("off")
        ax.set_title(f"{'CN' if lab == 0 else 'AD'} (CDR {df.cdr_folder[i]:g}) — {df.subject[i]}, s{df.slice[i]}", fontsize=6.5)
for ax in axes[2]:
    ax.axis("off")
for ax, (t, im) in zip(axes[2], panels):
    ax.imshow(im, cmap="gray"); ax.set_title(t, fontsize=8)
fig.suptitle("Rows 1–2: pre-processed CN and AD samples (centre slice).  Row 3: pre-processing steps for one image", fontsize=10)
fig.tight_layout()
FIGS = {"fig01": savefig(fig, "fig01_samples_and_preprocessing")}

# Figure S5: skull-stripping quality control on random slices (red = brain mask, yellow = fallback to head mask)
if SKULL_STRIP:
    qc = rng.choice(len(df), min(12, len(df)), replace=False)
    fig, axes = plt.subplots(2, 6, figsize=(14, 5.2))
    for ax, i in zip(axes.ravel(), qc):
        im0 = _resize(crop_pad_square(cv2.imread(df.path[i], cv2.IMREAD_GRAYSCALE)))
        im0 = apply_n4([im0])[0] if N4_ON else im0               # same order as the pipeline: N4, then stripping
        m0, ok0 = skull_strip(im0)
        ax.imshow(im0, cmap="gray")
        ax.contour(m0, levels=[0.5], colors=["#d62728" if ok0 else "#ffbf00"], linewidths=1.0)
        ax.set_title(f"{df.subject[i]}, s{df.slice[i]}" + ("" if ok0 else " (fallback)"), fontsize=7.5); ax.axis("off")
    for ax in axes.ravel()[len(qc):]:
        ax.axis("off")
    fig.suptitle(f"Figure S5. Skull-stripping QC (brain mask accepted on {PREPROC['strip_ok_pct']:.1f}% of all selected slices)", fontsize=10)
    fig.tight_layout()
    FIGS["figS5"] = savefig(fig, "figS5_skull_strip_qc")


## 5. Breaking the small dataset into parts
1. **Parts.** Subjects are split into `N_PARTS` disjoint parts with `StratifiedGroupKFold` (stratified by class, grouped by subject), so each part has about the same number of CN and AD subjects and no person is in two parts.
2. **Training / validation / test inside the parts.**
   - `PART_MODE = "independent"`: inside each part, subject-level `INNER_FOLDS`-fold CV. In every fold, ~`VAL_FRAC` of the training subjects are held out for early stopping / model selection. Each image of a part is tested exactly once, by models that never saw that subject.
   - `PART_MODE = "rotate"`: part *p* is the test set; the other parts (minus a validation share) are the training set.
3. The cell checks for leakage, shows the subjects and images of every split, and decides which parts to run now and which to load from an earlier session.

In [ ]:
def sgkf(y_, g_, n_splits, seed):
    """Stratified group K-fold; n_splits is reduced, if needed, to the number of groups in the smaller class."""
    n_min = min(len(np.unique(g_[y_ == c])) for c in (0, 1))
    n = max(2, min(int(n_splits), n_min))
    return list(StratifiedGroupKFold(n_splits=n, shuffle=True, random_state=seed).split(np.zeros(len(y_)), y_, g_)), n


def carve_val(tr_all, seed):
    """Hold out about VAL_FRAC of the training subjects (stratified, by group) for early stopping / model selection."""
    sp, _ = sgkf(y[tr_all], groups[tr_all], max(2, round(1 / VAL_FRAC)), seed)
    fit_rel, val_rel = sp[0]
    return tr_all[fit_rel], tr_all[val_rel]


# ---- 1) parts: disjoint, class-balanced sets of SUBJECTS --------------------------------------------
part_splits, N_PARTS_USED = sgkf(y, groups, N_PARTS, SEED)
if N_PARTS_USED != N_PARTS:
    print(f"NOTE: N_PARTS reduced from {N_PARTS} to {N_PARTS_USED} (too few subjects per class)")
PART_OF = np.full(len(y), -1)
for p_, (_, idx_) in enumerate(part_splits):
    PART_OF[idx_] = p_
PARTS = [np.where(PART_OF == p_)[0] for p_ in range(N_PARTS_USED)]
assert (PART_OF >= 0).all() and sum(len(a) for a in PARTS) == len(y)
for a in range(N_PARTS_USED):
    for b in range(a + 1, N_PARTS_USED):
        assert not set(groups[PARTS[a]]) & set(groups[PARTS[b]]), "a subject appears in two parts"

# ---- 2) training / validation / test sets ("tasks") of every part -----------------------------------------
TASKS = []
for p_, pidx in enumerate(PARTS):
    if PART_MODE == "independent":                       # subject-level CV INSIDE the part
        inner, n_in = sgkf(y[pidx], groups[pidx], INNER_FOLDS, SEED + 10 * p_)
        for k_, (tr_rel, te_rel) in enumerate(inner):
            tr_, va_ = carve_val(pidx[tr_rel], SEED + 10 * p_ + k_)
            TASKS.append(dict(part=p_, fold=k_, n_folds=n_in, train=tr_, val=va_, test=pidx[te_rel]))
    else:                                                # rotate: this part is the test set, the others train
        tr_all = np.concatenate([PARTS[q] for q in range(N_PARTS_USED) if q != p_])
        tr_, va_ = carve_val(tr_all, SEED + 10 * p_)
        TASKS.append(dict(part=p_, fold=0, n_folds=1, train=tr_, val=va_, test=pidx))

task_rows = []
for t in TASKS:
    g_tr, g_va, g_te = set(groups[t["train"]]), set(groups[t["val"]]), set(groups[t["test"]])
    assert not (g_tr & g_te) and not (g_va & g_te) and not (g_tr & g_va), "subject leakage between splits"
    if PART_MODE == "independent":
        assert set(PART_OF[np.concatenate([t["train"], t["val"], t["test"]])]) == {t["part"]}, "data from another part"
    cnt = lambda idx: f"{len(set(groups[idx]))} ({len(set(groups[idx][y[idx] == 0]))}/{len(set(groups[idx][y[idx] == 1]))})"
    task_rows.append({"Part": t["part"] + 1, "Inner fold": t["fold"] + 1 if PART_MODE == "independent" else "—",
                      "Train subjects (CN/AD)": cnt(t["train"]), "Val subjects (CN/AD)": cnt(t["val"]),
                      "Test subjects (CN/AD)": cnt(t["test"]), "Train images": len(t["train"]),
                      "Val images": len(t["val"]), "Test images": len(t["test"])})
TASK_TABLE = pd.DataFrame(task_rows)
tested = np.concatenate([t["test"] for t in TASKS])
assert sorted(tested.tolist()) == list(range(len(y))), "every image must be tested exactly once"
print(f"Leakage check PASSED: {len(TASKS)} train/val/test splits, no subject shared between training, validation and test.")

PART_TABLE = pd.DataFrame([{"Part": p_ + 1, "Subjects": len(set(groups[pidx])),
                            "CN subjects": len(set(groups[pidx][y[pidx] == 0])),
                            "AD subjects": len(set(groups[pidx][y[pidx] == 1])),
                            "Images": len(pidx), "CN images": int((y[pidx] == 0).sum()), "AD images": int((y[pidx] == 1).sum()),
                            "AD CDR levels": ", ".join(f"{c:g}: {n}" for c, n in df.iloc[pidx].drop_duplicates("subject")
                                                       .query("label == 1").cdr_folder.value_counts().sort_index().items()),
                            "Training data": ("this part only (inner CV)" if PART_MODE == "independent" else
                                              f"the other {N_PARTS_USED - 1} parts")}
                           for p_, pidx in enumerate(PARTS)])
SUBJECT_PARTS = (df.assign(part=PART_OF + 1).drop_duplicates("subject")
                 [["subject", "folder", "cdr_folder", "label", "part"]].sort_values(["part", "label", "subject"]))
PART_TABLE.to_csv(os.path.join(OUT_DIR, "tables", "parts_overview.csv"), index=False)
TASK_TABLE.to_csv(os.path.join(OUT_DIR, "tables", "parts_train_val_test_splits.csv"), index=False)
SUBJECT_PARTS.to_csv(os.path.join(OUT_DIR, "tables", "subjects_per_part.csv"), index=False)
print(f"PART_MODE = '{PART_MODE}':",
      f"each part is a separate experiment with {TASKS[0]['n_folds']}-fold subject-level CV inside it"
      if PART_MODE == "independent" else "each part is tested once by models trained on the other parts")
display(PART_TABLE)
display(TASK_TABLE)

# ---- 3) which parts to run now, which can be loaded from an earlier session ----------------------------------
RESULTS_DIR = os.path.join(OUT_DIR, "results")
_sig_cfg = {k: globals()[k] for k in (
    "PART_MODE", "N_PARTS", "INNER_FOLDS", "VAL_FRAC", "SEED", "SUBJECTS_PER_CLASS", "SLICES_PER_SUBJECT", "INPUT_MODE",
    "IMG_SIZE", "PREPROC_PRESET", "BIAS_CORRECTION", "SKULL_STRIP", "BRAIN_CROP", "DENOISE", "INTENSITY_NORM", "BACKBONE",
    "PRETRAINED", "CNN_EPOCHS", "N_PCA", "N_QUBITS", "ENCODING", "Q_EPOCHS", "Q_RESTARTS", "RUN_PROPOSED",
    "RUN_ML_BENCHMARK", "ML_PCA", "ML_TREES", "FAST_DEV_RUN")}
_sig_cfg.update(models=list(ML_MODELS), features={k: list(v) for k, v in ML_FEATURE_SETS.items()},
                ensemble=list(ENSEMBLE_MEMBERS), images=list(df.rel))
SIGNATURE = hashlib.md5(json.dumps(_sig_cfg, sort_keys=True, default=str).encode()).hexdigest()[:16]
want = (list(range(N_PARTS_USED)) if RUN_PARTS in ("all", None) else sorted({int(v) - 1 for v in RUN_PARTS}))
assert all(0 <= p_ < N_PARTS_USED for p_ in want), f"RUN_PARTS must be part numbers 1..{N_PARTS_USED}"


def saved_part(p_):
    """Path of a saved, matching result file for part p_ (or None)."""
    for d in [d_ for d_ in (RESUME_DIR, RESULTS_DIR) if d_]:
        meta = os.path.join(d, f"part{p_ + 1}_meta.json")
        if os.path.exists(meta):
            try:
                if json.load(open(meta)).get("signature") == SIGNATURE:
                    return d
            except Exception:
                pass
    return None


PART_STATUS = {p_: ("load" if RESUME and saved_part(p_) else "run" if p_ in want else "not run") for p_ in range(N_PARTS_USED)}
print("Part status:", {f"Part {p_ + 1}": s_ for p_, s_ in PART_STATUS.items()}, "| signature", SIGNATURE)


## 6. CNN feature extractor (Sec. 3.3)
VGG16 with batch normalisation, pre-trained on ImageNet and **fine-tuned separately in every training set** of every part (CN vs AD). The 4096-d fc7 output is the feature vector F_CNN; the ImageNet output layer is replaced by a 2-unit softmax. Training: AdamW with a 10× smaller learning rate for the pre-trained convolutional layers, the first `FREEZE_BLOCKS` blocks frozen, label smoothing, strong augmentation (training images only), checkpoint selection on the subject-level validation AUC and early stopping (`CNN_PATIENCE`, not before `CNN_MIN_EPOCHS`). Settings are those of the full run except fewer epochs, because each part is small.

In [ ]:
class ADCNN(nn.Module):
    def __init__(self, widths=CNN_WIDTHS, feat_dim=FEATURE_DIM, dropout=CNN_DROPOUT):
        super().__init__()
        layers, c = [], IN_CHANNELS
        for w in widths:
            layers += [nn.Conv2d(c, w, 3, padding=1, bias=False), nn.BatchNorm2d(w), nn.ReLU(),
                       nn.Conv2d(w, w, 3, padding=1, bias=False), nn.BatchNorm2d(w), nn.ReLU(), nn.MaxPool2d(2)]
            c = w
        self.body = nn.Sequential(*layers[:-1])   # up to the last conv block (Grad-CAM target layer)
        self.last_pool = layers[-1]
        self.gap = nn.AdaptiveAvgPool2d(2)
        self.fc1 = nn.Linear(c * 4, feat_dim)
        self.drop = nn.Dropout(dropout)
        self.fc2 = nn.Linear(feat_dim, 2)

    def embed_from_maps(self, A):
        return F.relu(self.fc1(torch.flatten(self.gap(self.last_pool(A)), 1)))

    def embed(self, x):
        return self.embed_from_maps(self.body(x))

    def forward(self, x):
        return self.fc2(self.drop(self.embed(x)))


class InputAdapter(nn.Module):
    """Greyscale input in [-1, 1] -> 3-channel input normalised with ImageNet statistics."""

    def __init__(self):
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, x):
        x = x * 0.5 + 0.5
        if x.shape[1] == 1:                       # 2D mode: grey slice -> 3 identical channels
            x = x.repeat(1, 3, 1, 1)
        return (x - self.mean) / self.std


class VGGFeatureCNN(nn.Module):
    """VGG16(-BN) with ImageNet weights; fc7 (4096-d) is the feature vector, a new 2-unit layer is the classifier."""

    def __init__(self, arch=BACKBONE, pretrained=True, dropout=CNN_DROPOUT):
        super().__init__()
        v = getattr(torchvision.models, arch)(weights="DEFAULT" if pretrained else None)
        feats = list(v.features.children())
        assert isinstance(feats[-1], nn.MaxPool2d)
        self.body = nn.Sequential(InputAdapter(), *feats[:-1])   # ends at the last conv ReLU (Grad-CAM target layer)
        self.last_pool, self.gap = feats[-1], v.avgpool
        self.fc6, self.fc7 = v.classifier[0], v.classifier[3]    # Linear(25088, 4096), Linear(4096, 4096)
        self.drop = nn.Dropout(dropout)
        self.fc2 = nn.Linear(self.fc7.out_features, 2)
        # freeze the first FREEZE_BLOCKS conv blocks (index in body just after the N-th max-pool)
        pools = [i for i, m in enumerate(self.body) if isinstance(m, nn.MaxPool2d)]
        self.n_frozen = pools[FREEZE_BLOCKS - 1] + 1 if pretrained and 0 < FREEZE_BLOCKS <= len(pools) else 0
        for m in list(self.body)[:self.n_frozen]:
            for q in m.parameters():
                q.requires_grad_(False)

    def train(self, mode=True):
        super().train(mode)
        for m in list(self.body)[:self.n_frozen]:      # frozen blocks keep their BatchNorm statistics
            m.eval()
        return self

    def head_parameters(self):
        return [p for m in (self.fc6, self.fc7, self.fc2) for p in m.parameters()]

    def embed_from_maps(self, A):
        h = self.drop(F.relu(self.fc6(torch.flatten(self.gap(self.last_pool(A)), 1))))
        return F.relu(self.fc7(h))                                # F_CNN: 4096-d

    def embed(self, x):
        return self.embed_from_maps(self.body(x))

    def forward(self, x):
        return self.fc2(self.drop(self.embed(x)))


ADCNN.head_parameters = lambda self: [p for m in (self.fc1, self.fc2) for p in m.parameters()]


def build_cnn(pretrained=None):
    if BACKBONE == "custom":
        return ADCNN()
    pre = PRETRAINED if pretrained is None else pretrained
    try:
        return VGGFeatureCNN(BACKBONE, pre)
    except Exception as e:
        raise SystemExit(f"Could not create {BACKBONE} with ImageNet weights ({type(e).__name__}: {e}).\n"
                         "Kaggle: Settings -> Internet ON (the weights, ~530 MB, are downloaded once), then re-run this cell.")


class IntensityAug:
    """Gamma, contrast and brightness jitter plus Gaussian noise, applied to tissue pixels only (background stays 0)."""

    def __call__(self, x):
        u = lambda a, b: float(torch.empty(1).uniform_(a, b))
        m = x > 0.02
        x = x.clamp(0, 1) ** math.exp(u(-0.2, 0.2))
        x = torch.where(m, (x - 0.5) * u(0.9, 1.1) + 0.5 + u(-0.05, 0.05), x)
        if u(0, 1) < 0.5:
            x = torch.where(m, x + torch.randn_like(x) * u(0, 0.03), x)
        return x.clamp(0, 1)


AUG = (T.Compose([T.RandomHorizontalFlip(),
                  T.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.9, 1.1), shear=5),
                  IntensityAug(),
                  T.RandomErasing(p=0.25, scale=(0.02, 0.08), value=0)])
       if AUGMENT == "strong" else
       T.Compose([T.RandomHorizontalFlip(), T.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.95, 1.05))]))


class MRIDataset(Dataset):
    def __init__(self, idx, augment):
        self.idx, self.augment = np.asarray(idx), augment

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        j = self.idx[i]
        img = torch.from_numpy(X3[j] if X3 is not None else X[j][None]).float().div(255)
        if self.augment:
            img = AUG(img)
        return (img - 0.5) / 0.5, int(y[j])


def to_batch(idx):
    arr = X3[idx] if X3 is not None else X[idx][:, None]
    return ((torch.from_numpy(arr).float().div(255) - 0.5) / 0.5).to(DEVICE)


@torch.no_grad()
def cnn_predict(model, idx, bs=64):
    model.eval()
    probs, feats, losses = [], [], []
    for s in range(0, len(idx), bs):
        b = idx[s:s + bs]
        xb = to_batch(b)
        f = model.embed(xb)
        logits = model.fc2(f)
        losses.append(F.cross_entropy(logits, torch.from_numpy(y[b]).to(DEVICE), reduction="sum").item())
        probs.append(torch.softmax(logits, 1)[:, 1].cpu().numpy())
        feats.append(f.cpu().numpy())
    return np.concatenate(probs), np.concatenate(feats).astype(np.float32), sum(losses) / len(idx)


AMP_ON = USE_AMP and DEVICE.type == "cuda"


def _grad_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=AMP_ON)
    except (AttributeError, TypeError):           # older PyTorch
        return torch.cuda.amp.GradScaler(enabled=AMP_ON)


def subject_auc(p, idx):
    """AUC of the mean slice probability per subject (group); 0.5 if only one class is present."""
    s_ = pd.DataFrame({"g": groups[idx], "p": p, "y": y[idx]}).groupby("g").mean()
    return float(roc_auc_score(s_.y, s_.p)) if s_.y.nunique() == 2 else 0.5


def train_cnn(fold, seed, verbose=True):
    set_seed(seed)
    model = build_cnn().to(DEVICE)
    head = model.head_parameters()
    head_ids = {id(q) for q in head}
    body = [q for q in model.parameters() if id(q) not in head_ids and q.requires_grad]
    groups_lr = ([{"params": body, "lr": CNN_LR_BACKBONE}, {"params": head, "lr": CNN_LR}]
                 if BACKBONE != "custom" and PRETRAINED else [{"params": list(model.parameters()), "lr": CNN_LR}])
    opt = torch.optim.AdamW(groups_lr, weight_decay=CNN_WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=2)
    scaler = _grad_scaler()
    g = torch.Generator(); g.manual_seed(seed)
    dl = DataLoader(MRIDataset(fold["train"], True), batch_size=BATCH_SIZE, shuffle=True,
                    num_workers=NUM_WORKERS, generator=g, drop_last=len(fold["train"]) % BATCH_SIZE == 1)
    best, best_vl, best_state, wait, hist = -float("inf"), float("inf"), None, 0, []
    t0 = time.time()
    for ep in range(CNN_EPOCHS):
        model.train()
        tl, tc, tn = 0.0, 0, 0
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with (torch.autocast(device_type="cuda", dtype=torch.float16) if AMP_ON else contextlib.nullcontext()):
                out = model(xb)
                loss = F.cross_entropy(out, yb, label_smoothing=LABEL_SMOOTHING)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tl += loss.item() * len(yb); tc += (out.argmax(1) == yb).sum().item(); tn += len(yb)
        pv, _, vl = cnn_predict(model, fold["val"])
        va = float(((pv >= 0.5) == y[fold["val"]]).mean())
        vauc = subject_auc(pv, fold["val"])
        lr_now = opt.param_groups[-1]["lr"]
        hist.append(dict(epoch=ep + 1, train_loss=tl / tn, train_acc=tc / tn, val_loss=vl, val_acc=va, val_auc=vauc, lr=lr_now))
        if verbose:
            print(f"  epoch {ep + 1:2d}  train loss {tl / tn:.4f} acc {tc / tn:.3f} | val loss {vl:.4f} acc {va:.3f} "
                  f"subject AUC {vauc:.3f} | lr {lr_now:.1e}")
        sched.step(vl)
        metric = vauc if SELECT_BY == "val_auc" else -vl
        improved = metric > best + 1e-4 or (SELECT_BY == "val_auc" and abs(metric - best) <= 1e-4 and vl < best_vl)
        if improved:
            best, best_vl, wait = metric, vl, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            best_epoch = ep + 1
        else:
            wait += 1
            if wait >= CNN_PATIENCE and ep + 1 >= CNN_MIN_EPOCHS:
                if verbose:
                    print(f"  early stopping at epoch {ep + 1} (kept epoch {best_epoch}: "
                          + (f"validation subject AUC {best:.3f})" if SELECT_BY == "val_auc" else f"validation loss {-best:.4f})"))
                break
    model.load_state_dict(best_state)
    hist = [dict(h_, kept=(h_["epoch"] == best_epoch)) for h_ in hist]
    if hist[-1]["train_acc"] < 0.65:
        print(f"  WARNING: training accuracy is only {hist[-1]['train_acc']:.2f} — the CNN has not learned the training "
              "data; downstream VQC/hybrid results will be at chance level.")
    return model, pd.DataFrame(hist), time.time() - t0


_tmp = build_cnn(pretrained=False)          # architecture only (no download) for parameter counts
N_PARAMS = {"cnn_total": sum(p.numel() for p in _tmp.parameters()),
            "cnn_conv": sum(p.numel() for p in _tmp.body.parameters()),
            "cnn_classification_part": sum(p.numel() for p in _tmp.head_parameters())}
if BACKBONE != "custom" and PRETRAINED:     # download the ImageNet weights once, before the folds start
    del _tmp
    _tmp = build_cnn()
    print(f"{BACKBONE}: ImageNet weights loaded")
N_PARAMS["cnn_trainable"] = sum(p.numel() for p in _tmp.parameters() if p.requires_grad)
with torch.no_grad():
    _a = _tmp.body(torch.zeros(1, IN_CHANNELS, IMG_SIZE, IMG_SIZE))
GRADCAM_MAP_SIZE = tuple(_a.shape[-2:])
print(_tmp)
print(N_PARAMS, "| Grad-CAM target maps:", tuple(_a.shape[1:]))
del _tmp, _a


## 7. Quantum circuits (Sec. 3.5–3.6) and verification against Qiskit
The circuits are defined with Qiskit's `zz_feature_map` (or RY angle encoding) and `real_amplitudes`. The output is the expectation value ⟨Z_k⟩ of every qubit (Eq. 13).

Qiskit runs in a **separate Python process** (a small worker script), because its compiled library and PyTorch's OpenMP library cannot share the limited "static TLS" memory of one process on recent Kaggle images. The same circuit is simulated exactly in PyTorch (state vector, complex64). This gives fast batched training and exact gradients, and it lets Grad-CAM back-propagate through the quantum layer. **Before any training, the PyTorch simulator is checked against Qiskit's `Statevector` on random inputs and parameters for every qubit count used.**

In this version the Qiskit cross-check is **optional** (`VERIFY_WITH_QISKIT`): if Qiskit cannot be installed, the notebook warns and continues with the PyTorch simulator.

In [ ]:
def entangling_pairs(n, kind=ENTANGLEMENT):
    if kind == "linear":
        return [(i, i + 1) for i in range(n - 1)]
    if kind == "full":
        return [(i, j) for i in range(n) for j in range(i + 1, n)]
    raise ValueError("ENTANGLEMENT must be 'linear' or 'full'")


_H = torch.tensor([[1, 1], [1, -1]], dtype=torch.complex64) / math.sqrt(2)


def _apply_1q(psi, U, k):
    psi = psi.movedim(k + 1, -1)
    if U.dim() == 2:
        psi = psi @ U.transpose(0, 1)
    else:
        shp = psi.shape
        psi = torch.einsum("bmj,bij->bmi", psi.reshape(shp[0], -1, 2), U).reshape(shp)
    return psi.movedim(-1, k + 1)


def _apply_phase(psi, lam, k):          # P(lam) = diag(1, e^{i lam}); lam has shape (B,)
    psi = psi.movedim(k + 1, -1)
    ph = torch.exp(1j * lam).reshape((-1,) + (1,) * (psi.dim() - 2))
    psi = torch.stack((psi[..., 0], psi[..., 1] * ph), dim=-1)
    return psi.movedim(-1, k + 1)


def _apply_cx(psi, c, t):
    psi = torch.movedim(psi, (c + 1, t + 1), (-2, -1))
    psi = torch.stack((psi[..., 0, :], psi[..., 1, :].flip(-1)), dim=-2)
    return torch.movedim(psi, (-2, -1), (c + 1, t + 1))


def _ry(theta):                          # theta scalar -> (2,2) or (B,) -> (B,2,2)
    c, s = torch.cos(theta / 2), torch.sin(theta / 2)
    U = torch.stack((torch.stack((c, -s), -1), torch.stack((s, c), -1)), -2)
    return torch.complex(U, torch.zeros_like(U))


class TorchVQC(nn.Module):
    """Exact state-vector simulation of qiskit_circuit(n); returns <Z_k> for every qubit k."""

    def __init__(self, n):
        super().__init__()
        self.n, self.pairs = n, entangling_pairs(n)
        self.theta = nn.Parameter(torch.empty(n * (ANSATZ_REPS + 1)).uniform_(-math.pi, math.pi))

    def forward(self, x):
        B, n = x.shape[0], self.n
        psi = torch.zeros((B, 2 ** n), dtype=torch.complex64, device=x.device)
        psi[:, 0] = 1
        psi = psi.reshape((B,) + (2,) * n)
        H = _H.to(x.device)
        if ENCODING == "zz":
            for _ in range(FM_REPS):
                for i in range(n):
                    psi = _apply_1q(psi, H, i)
                for i in range(n):
                    psi = _apply_phase(psi, 2 * x[:, i], i)
                for i, j in self.pairs:
                    psi = _apply_cx(psi, i, j)
                    psi = _apply_phase(psi, 2 * (math.pi - x[:, i]) * (math.pi - x[:, j]), j)
                    psi = _apply_cx(psi, i, j)
        else:
            for i in range(n):
                psi = _apply_1q(psi, _ry(x[:, i]), i)
        t = 0
        for _ in range(ANSATZ_REPS):
            for i in range(n):
                psi = _apply_1q(psi, _ry(self.theta[t]), i); t += 1
            for i, j in self.pairs:
                psi = _apply_cx(psi, i, j)
        for i in range(n):
            psi = _apply_1q(psi, _ry(self.theta[t]), i); t += 1
        prob = psi.real ** 2 + psi.imag ** 2
        out = []
        for k in range(n):
            pk = prob.movedim(k + 1, 1).reshape(B, 2, -1).sum(-1)
            out.append(pk[:, 0] - pk[:, 1])
        return torch.stack(out, 1)


def qiskit_expectations(n, Xq, theta):
    """<Z_k> of the Qiskit circuit (feature map + ansatz, Figure 7) via Statevector, computed in the Qiskit worker process."""
    return qiskit_call("expect", arrays=dict(X=np.asarray(Xq, dtype=np.float64), theta=np.asarray(theta, dtype=np.float64)),
                       n=int(n))["Z"]


def verify_simulator(n, n_samples=16, seed=0):
    r = np.random.default_rng(seed)
    Xq = r.uniform(0, np.pi, (n_samples, n)).astype(np.float32)
    m = TorchVQC(n)
    with torch.no_grad():
        theta = m.theta.numpy().astype(np.float64)
        ours = m(torch.from_numpy(Xq)).numpy()
    ref = qiskit_expectations(n, Xq.astype(np.float64), theta)
    return float(np.abs(ours - ref).max())


SIM_CHECK, CIRCUIT_INFO = {}, dict(qubits=N_QUBITS, encoding=ENCODING, feature_map_reps=FM_REPS if ENCODING == "zz" else 1,
                                   ansatz="RealAmplitudes", ansatz_reps=ANSATZ_REPS, entanglement=ENTANGLEMENT,
                                   trainable_parameters=N_QUBITS * (ANSATZ_REPS + 1))
if VERIFY_WITH_QISKIT and QISKIT_OK:
    SIM_CHECK = {n: verify_simulator(n) for n in sorted({N_QUBITS})}
    print("Max |<Z>_PyTorch - <Z>_Qiskit| on random inputs/parameters:", {k: f"{v:.1e}" for k, v in SIM_CHECK.items()})
    assert max(SIM_CHECK.values()) < 1e-4, "PyTorch simulator does not match Qiskit — do not continue"
    _png7 = os.path.join(OUT_DIR, "figures", "fig07_vqc_circuit.png")
    _info = qiskit_call("info", n=N_QUBITS, png=_png7,
                        title=f"VQC: {'ZZ feature map' if ENCODING == 'zz' else 'angle encoding'} + RealAmplitudes "
                              f"({N_QUBITS} qubits, {N_QUBITS * (ANSATZ_REPS + 1)} trainable parameters)")
    CIRCUIT_INFO.update(trainable_parameters=int(_info["n_theta"]), depth=int(_info["depth"]), cx_gates=int(_info["cx"]))
    if int(_info["drawn"]):
        display(IPImage(_png7))
    else:
        print("Circuit drawing (mpl) unavailable:", str(_info.get("draw_error", "")))
        print(str(_info.get("text", "")))
else:
    print("Qiskit cross-check skipped (VERIFY_WITH_QISKIT is False or Qiskit is unavailable); the PyTorch state-vector "
          "simulator above implements the same gates (H, P, CX, RY).")
print(CIRCUIT_INFO)


## 8. Classifier heads of the proposed pipeline (Sec. 3.6–3.7)

| Classifier | Input | Trainable parameters |
|---|---|---|
| **VQC** | leading *n* PCA components of fc7 scaled to [0, π] → VQC → ⟨Z⟩ (n values) → softmax | ansatz + (2n + 2) |
| **Hybrid CNN–VQC** (Eq. 14–15) | H = [standardised PCA components ⊕ ⟨Z⟩ of the VQC] → softmax | ansatz + (2(d + n) + 2) |
| Ablation: linear on *n* comps | same *n* PCA components → softmax (no quantum layer) | 2n + 2 |
| Ablation: linear on d comps | same standardised components → softmax (no quantum layer) | 2d + 2 |

d = `N_PCA` = 64, reduced automatically if a training set has fewer than 65 images. The two ablations show what the quantum layer adds on exactly the same inputs.

In [ ]:
class VQCClassifier(nn.Module):
    def __init__(self, n):
        super().__init__()
        self.q, self.out = TorchVQC(n), nn.Linear(n, 2)

    def forward(self, xq):
        return self.out(self.q(xq))


class HybridHead(nn.Module):
    def __init__(self, n, d=N_PCA):
        super().__init__()
        self.q, self.out = TorchVQC(n), nn.Linear(d + n, 2)

    def forward(self, zc, xq):
        return self.out(torch.cat([zc, self.q(xq)], 1))


class LinearHead(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.out = nn.Linear(d, 2)

    def forward(self, z):
        return self.out(z)


def _train_once(model, tr_inputs, ytr_t, va_inputs, yva_t, seed):
    model = model.to(Q_DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=Q_LR)
    best, best_state, wait, best_ep = float("inf"), None, 0, 0
    g = torch.Generator(); g.manual_seed(seed)
    ytr = ytr_t.cpu().numpy()
    for ep in range(Q_EPOCHS):
        model.train()
        perm = torch.randperm(len(ytr), generator=g)
        for s in range(0, len(ytr), Q_BATCH):
            b = perm[s:s + Q_BATCH]
            opt.zero_grad()
            loss = F.cross_entropy(model(*[a[b] for a in tr_inputs]), ytr_t[b])
            loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            vl = F.cross_entropy(model(*va_inputs), yva_t).item()
        if vl < best - 1e-5:
            best, wait, best_ep = vl, 0, ep + 1
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= Q_PATIENCE:
                break
    model.load_state_dict(best_state)
    model.best_epoch = best_ep                                   # epoch whose weights were kept (lowest validation loss)
    return model, best, ep + 1


def train_head(make_model, tr_inputs, ytr, va_inputs, yva, seed, restarts=None):
    """Train Q_RESTARTS randomly initialised copies; keep the one with the lowest validation loss."""
    t0 = time.time()
    tr_inputs = [torch.as_tensor(a, dtype=torch.float32, device=Q_DEVICE) for a in tr_inputs]
    va_inputs = [torch.as_tensor(a, dtype=torch.float32, device=Q_DEVICE) for a in va_inputs]
    ytr_t, yva_t = torch.as_tensor(ytr, device=Q_DEVICE), torch.as_tensor(yva, device=Q_DEVICE)
    best = None
    for r in range(restarts or Q_RESTARTS):
        set_seed(seed + 1000 * r)
        res = _train_once(make_model(), tr_inputs, ytr_t, va_inputs, yva_t, seed + 1000 * r)
        if best is None or res[1] < best[1]:
            best = res
    return best[0], time.time() - t0, best[2]


@torch.no_grad()
def head_predict(model, inputs):
    model.eval()
    dev = next(model.parameters()).device
    inputs = [torch.as_tensor(a, dtype=torch.float32, device=dev) for a in inputs]
    return torch.softmax(model(*inputs), 1)[:, 1].cpu().numpy()


def count_params(m):
    return sum(p.numel() for p in m.parameters())


## 9. Benchmark classifiers on frozen ImageNet features (Sec. 10b of the full run)
DenseNet121 (1024-d) and ResNet101 (2048-d) features are computed **once** for the small dataset by frozen ImageNet networks (no labels involved) and cached. In every training set: standardisation → PCA (`ML_PCA`, at most training images − 1) → one global scale, all fitted on the training + validation subjects only; then the 18 classifiers below with hyper-parameters fixed in advance. The LD-RNN, VQC and hybrid heads use the validation subjects for early stopping; the ensemble averages the probabilities of `ENSEMBLE_MEMBERS`.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier,
                              HistGradientBoostingClassifier)
from sklearn.neural_network import MLPClassifier

try:
    from xgboost import XGBClassifier
except Exception:
    XGBClassifier = None
try:
    from lightgbm import LGBMClassifier
except Exception:
    LGBMClassifier = None

# ---- names, families and inputs of every classifier ----------------------------------------------------
PROPOSED_NAMES = {"cnn": "CNN (fine-tuned VGG16-BN)", "vqc": f"VQC ({N_QUBITS} qubits, proposed)",
                  "hybrid": "Hybrid CNN–VQC (proposed)", "lin_n": f"Linear on {N_QUBITS} PCA comps (ablation)",
                  "lin64": f"Linear on {N_PCA} PCA comps (ablation)"}
ZOO_NAMES = {"logreg": "Logistic regression", "lda": "LDA (shrinkage)", "gnb": "Gaussian naive Bayes",
             "knn": "k-NN (k = 15)", "dtree": "Decision tree", "svm_linear": "Linear SVM", "svm_rbf": "RBF SVM",
             "rf": "Random forest", "extra_trees": "Extra trees", "adaboost": "AdaBoost", "hgb": "Gradient boosting",
             "xgboost": "XGBoost", "lightgbm": "LightGBM", "mlp": "MLP (256–64)", "ldrnn": "LD-RNN (LSTM)",
             "vqc_dual": f"VQC head ({N_QUBITS} qubits, frozen feats)", "hybrid_dual": "Hybrid VQC head (frozen feats)",
             "ensemble": "Soft-voting ensemble"}
ZOO_FAMILY = {**{k: "Classical ML" for k in ("logreg", "lda", "gnb", "knn", "dtree", "svm_linear", "svm_rbf", "rf",
                                              "extra_trees", "adaboost", "hgb", "xgboost", "lightgbm")},
              "mlp": "Neural (MLP / LSTM)", "ldrnn": "Neural (MLP / LSTM)", "vqc_dual": "Quantum heads",
              "hybrid_dual": "Quantum heads", "ensemble": "Ensemble"}
FAMILY_COLORS = {"Proposed (fine-tuned VGG16-BN)": COLORS["hybrid"], "Classical ML": "#8a8984",
                 "Neural (MLP / LSTM)": "#52514e", "Quantum heads": COLORS["vqc"], "Ensemble": COLORS["cnn"]}
ZOO_INPUT = {"ldrnn": "PCA comps → 512 projection, LSTM(128), dense 128 → 64, dropout 0.3, Adam 1e-3, early stopping on val",
             "vqc_dual": f"leading {N_QUBITS} PCA comps → [0, π] → ZZ map + RealAmplitudes → softmax",
             "hybrid_dual": f"{N_PCA} standardised PCA comps ⊕ ⟨Z⟩ of {N_QUBITS} qubits → softmax",
             "ensemble": "mean probability of " + ", ".join(ENSEMBLE_MEMBERS)}


class DecisionSigmoid:
    """SVM scores as probabilities: p = 1 / (1 + exp(-decision value)); 0.5 equals the SVM's own decision rule."""

    def __init__(self, est):
        self.est = est

    def fit(self, X_, y_):
        self.est.fit(X_, y_)
        return self

    def predict_proba(self, X_):
        p = 1.0 / (1.0 + np.exp(-np.clip(self.est.decision_function(X_), -30, 30)))
        return np.column_stack([1 - p, p])


def make_classifier(key, seed, n_fit=10**6):
    """Classical classifiers with hyper-parameters fixed in advance (no tuning on test data)."""
    if key == "logreg":
        return LogisticRegression(C=0.05, max_iter=5000)
    if key == "lda":
        return LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")
    if key == "gnb":
        return GaussianNB()
    if key == "knn":
        return KNeighborsClassifier(n_neighbors=int(min(15, max(1, n_fit - 1))), weights="distance")
    if key == "dtree":
        return DecisionTreeClassifier(max_depth=8, min_samples_leaf=10, random_state=seed)
    if key == "svm_linear":
        return DecisionSigmoid(LinearSVC(C=0.01, max_iter=20000))
    if key == "svm_rbf":
        return DecisionSigmoid(SVC(kernel="rbf", C=1.0, gamma="scale"))
    if key == "rf":
        return RandomForestClassifier(n_estimators=ML_TREES, min_samples_leaf=2, n_jobs=-1, random_state=seed)
    if key == "extra_trees":
        return ExtraTreesClassifier(n_estimators=ML_TREES, min_samples_leaf=2, n_jobs=-1, random_state=seed)
    if key == "adaboost":
        return AdaBoostClassifier(n_estimators=200, learning_rate=0.5, random_state=seed)
    if key == "hgb":
        return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=seed)
    if key == "xgboost" and XGBClassifier is not None:
        return XGBClassifier(n_estimators=400, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.5,
                             tree_method="hist", eval_metric="logloss", random_state=seed, n_jobs=-1)
    if key == "lightgbm" and LGBMClassifier is not None:
        return LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31, subsample=0.8, subsample_freq=1,
                              colsample_bytree=0.5, random_state=seed, verbose=-1, n_jobs=-1)
    if key == "mlp":
        return MLPClassifier(hidden_layer_sizes=(256, 64), alpha=1e-3, early_stopping=n_fit >= 40, max_iter=300,
                             random_state=seed)
    return None


ZOO_PARAMS = {"logreg": "C = 0.05 (L2)", "lda": "lsqr solver, Ledoit–Wolf shrinkage", "gnb": "default",
              "knn": "k = 15, distance weights", "dtree": "max depth 8, min 10 per leaf", "svm_linear": "C = 0.01",
              "svm_rbf": "C = 1, γ = scale", "rf": f"{ML_TREES} trees, min 2 per leaf",
              "extra_trees": f"{ML_TREES} trees, min 2 per leaf", "adaboost": "200 stumps, learning rate 0.5",
              "hgb": "300 iterations, learning rate 0.05", "xgboost": "400 trees, depth 4, lr 0.05, subsample 0.8",
              "lightgbm": "400 trees, 31 leaves, lr 0.05, subsample 0.8", "mlp": "256–64 hidden units, α = 1e-3",
              **ZOO_INPUT}


class LDRNN(nn.Module):
    """LD-RNN classifier (Thatere & Reddy, Eq. 6-17): projection -> LSTM(128) -> dense 128 -> 64 -> softmax."""

    def __init__(self, d_in, n_cls=2):
        super().__init__()
        self.proj = nn.Linear(d_in, 512)
        self.lstm = nn.LSTM(512, 128, batch_first=True)
        self.drop1, self.drop2 = nn.Dropout(0.3), nn.Dropout(0.3)
        self.fc1, self.fc2, self.out = nn.Linear(128, 128), nn.Linear(128, 64), nn.Linear(64, n_cls)

    def forward(self, x):
        h = F.relu(self.proj(x)).unsqueeze(1)            # the PCA vector as one time step (as in the paper)
        _, (hT, _) = self.lstm(h)
        z = F.relu(self.fc1(self.drop1(hT[-1])))
        return self.out(self.drop2(F.relu(self.fc2(z))))


def train_ldrnn(Ztr, ytr, Zva, yva, seed):
    set_seed(seed)
    m = LDRNN(Ztr.shape[1]).to(DEVICE)
    opt = torch.optim.Adam(m.parameters(), lr=LDRNN_LR)
    xt, yt_ = torch.tensor(Ztr, device=DEVICE), torch.tensor(ytr, device=DEVICE)
    xv, yv_ = torch.tensor(Zva, device=DEVICE), torch.tensor(yva, device=DEVICE)
    g = torch.Generator(); g.manual_seed(seed)
    best, best_state, wait = float("inf"), None, 0
    for ep in range(LDRNN_EPOCHS):
        m.train()
        perm = torch.randperm(len(yt_), generator=g)
        for s in range(0, len(perm), LDRNN_BATCH):
            b = perm[s:s + LDRNN_BATCH].to(DEVICE)
            if len(b) < 2:
                continue
            opt.zero_grad()
            F.cross_entropy(m(xt[b]), yt_[b]).backward()
            opt.step()
        m.eval()
        with torch.no_grad():
            vl = F.cross_entropy(m(xv), yv_).item()
        if vl < best - 1e-5 or best_state is None:
            best, wait, best_state = vl, 0, {k: v.detach().clone() for k, v in m.state_dict().items()}
        else:
            wait += 1
            if wait >= LDRNN_PATIENCE:
                break
    m.load_state_dict(best_state)
    return m


@torch.no_grad()
def ldrnn_predict(m, Z):
    m.eval()
    return torch.softmax(m(torch.tensor(Z, device=DEVICE)), 1)[:, 1].cpu().numpy()


def build_frozen(arch):
    net = getattr(torchvision.models, arch)(weights="DEFAULT" if PRETRAINED else None)
    if arch.startswith("densenet"):
        body = nn.Sequential(net.features, nn.ReLU(inplace=True), nn.AdaptiveAvgPool2d(1), nn.Flatten())
    elif arch.startswith("vit") or arch.startswith("swin"):
        for attr in ("heads", "head"):
            if hasattr(net, attr):
                setattr(net, attr, nn.Identity())
        body = net
    else:                                                    # ResNet family: everything up to global average pooling
        body = nn.Sequential(*list(net.children())[:-1], nn.Flatten())
    return nn.Sequential(InputAdapter(), body).to(DEVICE).eval()


@torch.no_grad()
def extract_features(model, idx, bs=48):
    out = []
    for s in range(0, len(idx), bs):
        xb = to_batch(idx[s:s + bs])
        if xb.shape[-1] != 224:                              # ImageNet backbones (ViT in particular) expect 224 x 224
            xb = F.interpolate(xb, size=(224, 224), mode="bilinear", align_corners=False)
        with (torch.autocast(device_type="cuda", dtype=torch.float16) if AMP_ON else contextlib.nullcontext()):
            out.append(model(xb).float().cpu().numpy())
    return np.concatenate(out).astype(np.float32)


def make_reducer(Fz_, fit_idx, seed):
    """Standardise -> PCA (ML_PCA comps, capped at n - 1, not whitened) -> one global SD; fitted on fit_idx only."""
    sc = StandardScaler().fit(Fz_[fit_idx])
    n_comp = int(min(ML_PCA, len(fit_idx) - 1, Fz_.shape[1]))
    pca = PCA(n_components=n_comp, random_state=seed).fit(sc.transform(Fz_[fit_idx]))
    g = float(pca.transform(sc.transform(Fz_[fit_idx])).std()) + 1e-8
    return lambda idx: (pca.transform(sc.transform(Fz_[idx])) / g).astype(np.float32)


# ---- which classifiers run --------------------------------------------------------------------------------
ACTIVE = [k for k in ML_MODELS if k in ZOO_NAMES] if RUN_ML_BENCHMARK else []
for _k, _cls in (("xgboost", XGBClassifier), ("lightgbm", LGBMClassifier)):
    if _k in ACTIVE and _cls is None:
        print(f"{_k} is not installed -> skipped"); ACTIVE.remove(_k)
ENS = [k for k in ENSEMBLE_MEMBERS if k in ACTIVE and k != "ensemble"] or [k for k in ("hgb",) if k in ACTIVE]
if "lightgbm" in ENSEMBLE_MEMBERS and "lightgbm" not in ACTIVE and "hgb" in ACTIVE and "hgb" not in ENS:
    ENS.append("hgb")                                        # same-family substitute, decided by availability only
MULTI_FS = len(ML_FEATURE_SETS) > 1
zkey = lambda key, fs: key if not MULTI_FS else f"{key}|{fs}"

MODEL_KEYS, MODEL_NAME, MODEL_FAMILY, MODEL_INPUT = [], {}, {}, {}
if RUN_PROPOSED:
    for k in ("cnn", "vqc", "hybrid", "lin_n", "lin64"):
        MODEL_KEYS.append(k); MODEL_NAME[k] = PROPOSED_NAMES[k]; MODEL_FAMILY[k] = "Proposed (fine-tuned VGG16-BN)"
    MODEL_INPUT.update(cnn="2.5D slices → VGG16-BN fine-tuned (fc7, 4096-d) → softmax",
                       vqc=f"fc7 → PCA → leading {N_QUBITS} comps → [0, π] → ZZ map + RealAmplitudes → softmax",
                       hybrid=f"[{N_PCA} standardised PCA comps of fc7 ⊕ ⟨Z⟩ of the VQC] → softmax",
                       lin_n=f"same {N_QUBITS} PCA comps → softmax (no quantum layer)",
                       lin64=f"same {N_PCA} standardised PCA comps → softmax (no quantum layer)")
for fs_name, archs in (ML_FEATURE_SETS.items() if RUN_ML_BENCHMARK else []):
    for k in ACTIVE:
        mk = zkey(k, fs_name)
        MODEL_KEYS.append(mk)
        MODEL_NAME[mk] = ZOO_NAMES[k] + (f" [{fs_name}]" if MULTI_FS else "")
        MODEL_FAMILY[mk] = ZOO_FAMILY[k]
        MODEL_INPUT[mk] = f"{' + '.join(archs)} (frozen ImageNet) → PCA; " + ZOO_PARAMS.get(k, "")
assert MODEL_KEYS, "Nothing to run: set RUN_PROPOSED and/or RUN_ML_BENCHMARK to True"
print(f"{len(MODEL_KEYS)} classifiers:", ", ".join(MODEL_NAME[k] for k in MODEL_KEYS))

# ---- frozen ImageNet features: computed ONCE for the small dataset (no labels used), cached on disk --------------
FROZEN, FEAT_INFO = {}, {}


def ensure_frozen():
    """Compute (or load from the cache) the frozen features of every feature set, once."""
    for fs_name, archs in ML_FEATURE_SETS.items():
        if fs_name in FROZEN:
            continue
        cache = os.path.join(OUT_DIR, "cache", f"frozen_{fs_name}_{SIGNATURE}.npy")
        t0 = time.time()
        if os.path.exists(cache):
            FROZEN[fs_name] = np.load(cache)
        else:
            parts_ = []
            for arch in archs:
                try:
                    fm = build_frozen(arch)
                except Exception as e:
                    raise SystemExit(f"Could not load {arch} with ImageNet weights ({type(e).__name__}: {e}). "
                                     "Kaggle: Internet ON.")
                parts_.append(extract_features(fm, np.arange(len(y)))); del fm
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
            FROZEN[fs_name] = np.concatenate(parts_, 1)
            np.save(cache, FROZEN[fs_name])
        FEAT_INFO[fs_name] = dict(backbones=" + ".join(archs), total=FROZEN[fs_name].shape[1], minutes=(time.time() - t0) / 60)
        print(f"[{fs_name}] frozen features {FROZEN[fs_name].shape} ({FEAT_INFO[fs_name]['backbones']}) "
              f"in {FEAT_INFO[fs_name]['minutes']:.1f} min")


if RUN_ML_BENCHMARK and any(s_ == "run" for s_ in PART_STATUS.values()):
    ensure_frozen()


## 10. Metrics, and training every classifier in every part
**Metrics** (threshold 0.5 on P(AD); AD is the positive class): accuracy; balanced accuracy; precision (PPV); sensitivity (recall); specificity; NPV; F1; Matthews correlation coefficient (MCC); Cohen's κ; ROC AUC; Brier score; log loss; TP/FN/TN/FP. **95% CIs** for accuracy and AUC come from a subject-level bootstrap (slices of one person are resampled together, stratified by class). **Subject-level** results average each person's slice probabilities. In independent mode, the spread over the inner folds of a part is reported too.

In [ ]:
from sklearn.metrics import matthews_corrcoef, cohen_kappa_score, brier_score_loss, log_loss, classification_report


def detailed_metrics(yt, p, thr=0.5):
    """All threshold (0.5) and ranking metrics of one set of predictions. Percentages for rates, AUC in [0, 1]."""
    yt, p = np.asarray(yt).astype(int), np.asarray(p, dtype=float)
    yh = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(yt, yh, labels=[0, 1]).ravel()
    div = lambda a, b: 100 * a / b if b else np.nan
    two = len(np.unique(yt)) == 2
    rec, spec = div(tp, tp + fn), div(tn, tn + fp)
    return dict(n=len(yt), accuracy=div(tp + tn, len(yt)), balanced_accuracy=float(np.nanmean([rec, spec])),
                precision=div(tp, tp + fp), recall=rec, specificity=spec, npv=div(tn, tn + fn),
                f1=div(2 * tp, 2 * tp + fp + fn),
                mcc=float(matthews_corrcoef(yt, yh)) if two else np.nan,
                kappa=float(cohen_kappa_score(yt, yh)) if two else np.nan,
                auc=float(roc_auc_score(yt, p)) if two else np.nan,
                brier=float(brier_score_loss(yt, p)),
                log_loss=float(log_loss(yt, np.clip(p, 1e-7, 1 - 1e-7), labels=[0, 1])),
                tp=int(tp), fp=int(fp), tn=int(tn), fn=int(fn))


def group_bootstrap_ci(yt, p, g, fn, B=N_BOOTSTRAP, seed=SEED):
    """95% CI by resampling SUBJECTS (stratified by class), so correlated slices of one person stay together."""
    r = np.random.default_rng(seed)
    ug, inv = np.unique(g, return_inverse=True)
    members = [np.where(inv == i)[0] for i in range(len(ug))]
    glab = np.array([yt[m[0]] for m in members])
    pos, neg = np.where(glab == 1)[0], np.where(glab == 0)[0]
    if len(pos) == 0 or len(neg) == 0:
        return np.array([np.nan, np.nan])
    vals = []
    for _ in range(B):
        idx = np.concatenate([members[i] for i in np.concatenate([r.choice(pos, len(pos)), r.choice(neg, len(neg))])])
        vals.append(fn(yt[idx], p[idx]))
    return np.percentile(vals, [2.5, 97.5])


def subject_level(idx, p):
    """Average the slice probabilities of each subject -> one prediction per subject."""
    sb = pd.DataFrame({"g": df.subject.values[idx], "y": y[idx], "p": p}).groupby("g").mean()
    return sb.y.values.astype(int), sb.p.values


_acc = lambda yt, p_: 100 * float(np.mean((np.asarray(p_) >= 0.5) == np.asarray(yt)))
_auc = lambda yt, p_: float(roc_auc_score(yt, p_)) if len(np.unique(yt)) == 2 else np.nan


def quick_part_table(p_):
    """Short table printed as soon as a part is finished (the detailed tables follow in Section 11)."""
    idx = PARTS[p_]
    rows = []
    for m in MODEL_KEYS:
        pr = PRED[m][idx]
        if not np.isfinite(pr).all():
            continue
        q = detailed_metrics(y[idx], pr)
        ys_, ps_ = subject_level(idx, pr)
        rows.append({"Classifier": MODEL_NAME[m], "Accuracy (%)": round(q["accuracy"], 1), "AUC": round(q["auc"], 3),
                     "Sensitivity (%)": round(q["recall"], 1), "Specificity (%)": round(q["specificity"], 1),
                     "F1 (%)": round(q["f1"], 1), "Subject-level accuracy (%)": round(_acc(ys_, ps_), 1)})
    return pd.DataFrame(rows)


### Training loop
For every part and every inner fold (or every rotated part): fine-tune the CNN → extract fc7 features → PCA/scalers on the training images → VQC, hybrid head and both ablations; then the 18 benchmark classifiers on the frozen features. One line per fold shows the test accuracy of every classifier (keys as in `MODEL_KEYS`), and a short table follows each part. **Each part is saved to `results/` as soon as it is finished**, so a session that stops keeps the completed parts.

In [ ]:
import shutil

PRED = {m: np.full(len(y), np.nan) for m in MODEL_KEYS}   # out-of-fold P(AD) of every classifier for every image
FOLD_OF = np.full(len(y), -1)
TIME_ROWS, HIST_ALL, PART_DONE, ERRORS = [], [], [], []


def run_task(t):
    """Train every classifier on t['train'] (+ t['val'] for model selection) and predict t['test']."""
    p_, k_, tr, va, te = t["part"], t["fold"], t["train"], t["val"], t["test"]
    seed = SEED + 1000 * p_ + 100 * k_
    y_tr, y_va = y[tr], y[va]
    out, times, info = {}, {}, ""
    if RUN_PROPOSED:
        # 1) CNN fine-tuned on this training set; early stopping on the validation subjects
        cnn, hist, t_cnn = train_cnn(dict(train=tr, val=va), seed, verbose=VERBOSE_CNN)
        kept = int(hist.loc[hist.kept, "epoch"].iloc[0]) if hist.kept.any() else len(hist)
        HIST_ALL.append(hist.assign(part=p_ + 1, fold=k_ + 1))
        _, F_tr, _ = cnn_predict(cnn, tr)
        _, F_va, _ = cnn_predict(cnn, va)
        out["cnn"], F_te, _ = cnn_predict(cnn, te)
        times["cnn"] = t_cnn
        del cnn
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        # 2) PCA + scalers on the training images only -> quantum / linear heads
        n_pca = int(min(N_PCA, len(tr) - 1, F_tr.shape[1]))
        pca = PCA(n_components=n_pca, random_state=seed).fit(F_tr)
        Z_tr, Z_va, Z_te = (pca.transform(a) for a in (F_tr, F_va, F_te))
        sc = StandardScaler().fit(Z_tr)
        C_tr, C_va, C_te = (sc.transform(a) for a in (Z_tr, Z_va, Z_te))
        mm = MinMaxScaler((0, np.pi)).fit(Z_tr[:, :N_QUBITS])
        q_tr, q_va, q_te = (np.clip(mm.transform(a[:, :N_QUBITS]), 0, np.pi) for a in (Z_tr, Z_va, Z_te))
        heads = {"vqc": (lambda: VQCClassifier(N_QUBITS), [q_tr], [q_va], [q_te]),
                 "hybrid": (lambda: HybridHead(N_QUBITS, d=n_pca), [C_tr, q_tr], [C_va, q_va], [C_te, q_te]),
                 "lin_n": (lambda: LinearHead(N_QUBITS), [q_tr], [q_va], [q_te]),
                 "lin64": (lambda: LinearHead(n_pca), [C_tr], [C_va], [C_te])}
        for j, (m_, (make, a_tr, a_va, a_te)) in enumerate(heads.items()):
            head, t_h, _ = train_head(make, a_tr, y_tr, a_va, y_va, seed + 1 + j)
            out[m_], times[m_] = head_predict(head, a_te), t_h
        info = (f"CNN: {len(hist)} epochs (kept {kept}), train acc {hist.train_acc.iloc[-1]:.2f}, "
                f"val subject AUC {hist.loc[hist.kept, 'val_auc'].iloc[0] if hist.kept.any() else float('nan'):.2f}")
    for fs_name, Fz in FROZEN.items():
        # 3) benchmark classifiers on frozen features: scaling + PCA fitted on training + validation subjects only
        fit = np.concatenate([tr, va])
        Zf = make_reducer(Fz, fit, seed)
        Z_fit, Z_tr, Z_va, Z_te = Zf(fit), Zf(tr), Zf(va), Zf(te)
        for key in ACTIVE:
            if key in ("ldrnn", "vqc_dual", "hybrid_dual", "ensemble"):
                continue
            t1 = time.time()
            try:
                clf = make_classifier(key, seed, n_fit=len(fit)).fit(Z_fit, y[fit])
                out[zkey(key, fs_name)] = clf.predict_proba(Z_te)[:, 1]
            except Exception as e:                   # one failing library must not stop the run
                ERRORS.append(f"part {p_ + 1} fold {k_ + 1} {key}: {type(e).__name__}: {e}")
            times[zkey(key, fs_name)] = time.time() - t1
        if "ldrnn" in ACTIVE:                        # LSTM: early stopping on the validation subjects
            t1 = time.time()
            out[zkey("ldrnn", fs_name)] = ldrnn_predict(train_ldrnn(Z_tr, y_tr, Z_va, y_va, seed + 31), Z_te)
            times[zkey("ldrnn", fs_name)] = time.time() - t1
        if "vqc_dual" in ACTIVE or "hybrid_dual" in ACTIVE:
            n_q = int(min(N_PCA, len(tr) - 1))
            pq = PCA(n_components=n_q, random_state=seed).fit(Fz[tr])
            Q_tr, Q_va, Q_te = (pq.transform(Fz[i_]) for i_ in (tr, va, te))
            scq = StandardScaler().fit(Q_tr)
            mmq = MinMaxScaler((0, np.pi)).fit(Q_tr[:, :N_QUBITS])
            qs = [np.clip(mmq.transform(a[:, :N_QUBITS]), 0, np.pi) for a in (Q_tr, Q_va, Q_te)]
            cs = [scq.transform(a) for a in (Q_tr, Q_va, Q_te)]
            if "vqc_dual" in ACTIVE:
                t1 = time.time()
                mq, _, _ = train_head(lambda: VQCClassifier(N_QUBITS), [qs[0]], y_tr, [qs[1]], y_va, seed + 41)
                out[zkey("vqc_dual", fs_name)], times[zkey("vqc_dual", fs_name)] = head_predict(mq, [qs[2]]), time.time() - t1
            if "hybrid_dual" in ACTIVE:
                t1 = time.time()
                mh, _, _ = train_head(lambda: HybridHead(N_QUBITS, d=n_q), [cs[0], qs[0]], y_tr, [cs[1], qs[1]], y_va,
                                      seed + 42)
                out[zkey("hybrid_dual", fs_name)] = head_predict(mh, [cs[2], qs[2]])
                times[zkey("hybrid_dual", fs_name)] = time.time() - t1
        if "ensemble" in ACTIVE:
            mem = [zkey(m_, fs_name) for m_ in ENS if zkey(m_, fs_name) in out]
            if mem:
                out[zkey("ensemble", fs_name)] = np.mean([out[m_] for m_ in mem], 0)
                times[zkey("ensemble", fs_name)] = float(sum(times[m_] for m_ in mem))
    return out, times, info


def save_part(p_):
    idx = PARTS[p_]
    pd.DataFrame({"rel": df.rel.values[idx], "subject": df.subject.values[idx], "label": y[idx], "part": p_ + 1,
                  "fold": FOLD_OF[idx] + 1, **{m: PRED[m][idx] for m in MODEL_KEYS}}
                 ).to_csv(os.path.join(RESULTS_DIR, f"part{p_ + 1}_predictions.csv"), index=False)
    pd.DataFrame([r for r in TIME_ROWS if r["part"] == p_ + 1]).to_csv(
        os.path.join(RESULTS_DIR, f"part{p_ + 1}_times.csv"), index=False)
    h = [h_ for h_ in HIST_ALL if int(h_.part.iloc[0]) == p_ + 1]
    if h:
        pd.concat(h).to_csv(os.path.join(RESULTS_DIR, f"part{p_ + 1}_cnn_history.csv"), index=False)
    with open(os.path.join(RESULTS_DIR, f"part{p_ + 1}_meta.json"), "w") as fh:   # written last = "part complete"
        json.dump(dict(signature=SIGNATURE, part=p_ + 1, models=MODEL_KEYS, env=ENV,
                       finished=datetime.datetime.now().isoformat(timespec="seconds")), fh, indent=1)


def load_part(p_):
    """Load a part saved by an earlier session; returns False if the files do not match this run."""
    d = saved_part(p_)
    try:
        pr = pd.read_csv(os.path.join(d, f"part{p_ + 1}_predictions.csv"))
        idx = PARTS[p_]
        if list(pr.rel) != list(df.rel.values[idx]) or not set(MODEL_KEYS) <= set(pr.columns):
            print(f"  Part {p_ + 1}: saved file does not match the current images/classifiers -> re-run"); return False
        for m in MODEL_KEYS:
            PRED[m][idx] = pr[m].values
        FOLD_OF[idx] = pr.fold.values - 1
        tf = os.path.join(d, f"part{p_ + 1}_times.csv")
        if os.path.exists(tf) and os.path.getsize(tf) > 1:
            TIME_ROWS.extend(pd.read_csv(tf).to_dict("records"))
        hf = os.path.join(d, f"part{p_ + 1}_cnn_history.csv")
        if os.path.exists(hf):
            HIST_ALL.append(pd.read_csv(hf))
    except Exception as e:
        print(f"  Part {p_ + 1}: could not load ({type(e).__name__}: {e}) -> re-run"); return False
    if os.path.abspath(d) != os.path.abspath(RESULTS_DIR):          # keep a copy with this run's outputs
        for f_ in os.listdir(d):
            if f_.startswith(f"part{p_ + 1}_"):
                shutil.copy(os.path.join(d, f_), RESULTS_DIR)
    return True


T_RUN = time.time()
for p_ in range(N_PARTS_USED):
    pidx = PARTS[p_]
    head_ = (f"Part {p_ + 1}/{N_PARTS_USED}: {len(set(groups[pidx]))} subjects "
             f"({len(set(groups[pidx][y[pidx] == 0]))} CN / {len(set(groups[pidx][y[pidx] == 1]))} AD), {len(pidx)} images")
    if PART_STATUS[p_] == "load":
        if load_part(p_):
            PART_DONE.append(p_); print(f"=== {head_} — loaded from {saved_part(p_)} ==="); continue
        PART_STATUS[p_] = "run" if p_ in want else "not run"
    if PART_STATUS[p_] == "not run":
        print(f"=== {head_} — not run in this session (RUN_PARTS = {RUN_PARTS}) ==="); continue
    print(f"\n=== {head_} ===")
    if RUN_ML_BENCHMARK:
        ensure_frozen()                              # no-op if the features are already there
    t_part = time.time()
    for t in [t_ for t_ in TASKS if t_["part"] == p_]:
        out, times, info = run_task(t)
        te = t["test"]
        for m, pr in out.items():
            PRED[m][te] = pr
        FOLD_OF[te] = t["fold"]
        TIME_ROWS.extend(dict(part=p_ + 1, fold=t["fold"] + 1, model=m, seconds=s_) for m, s_ in times.items())
        where = f"inner fold {t['fold'] + 1}/{t['n_folds']}" if PART_MODE == "independent" else f"test = part {p_ + 1}"
        print(f"  {where}: test {len(set(groups[te]))} subjects / {len(te)} images" + (f" | {info}" if info else ""))
        print("    test accuracy %: " + ", ".join(f"{m} {_acc(y[te], out[m]):.0f}" for m in MODEL_KEYS if m in out))
    save_part(p_)
    PART_DONE.append(p_)
    print(f"  Part {p_ + 1} finished in {(time.time() - t_part) / 60:.1f} min; saved to {RESULTS_DIR}")
    display(quick_part_table(p_))

PART_DONE = sorted(PART_DONE)
print(f"\nParts with results: {[p_ + 1 for p_ in PART_DONE]} of {N_PARTS_USED} "
      f"({(time.time() - T_RUN) / 60:.1f} min in this session)")
if ERRORS:
    print(f"{len(ERRORS)} classifier fits failed (their predictions are missing in those folds):", *ERRORS[:10], sep="\n  ")
assert PART_DONE, "No part has results yet: set RUN_PARTS to the parts to run in this session."


In [ ]:
acc_fn = lambda a, b: np.mean((b >= 0.5) == a)
T_DF = pd.DataFrame(TIME_ROWS) if TIME_ROWS else pd.DataFrame(columns=["part", "fold", "model", "seconds"])
rows, fold_rows, MISSING = [], [], []
t0 = time.time()
for p_ in PART_DONE + ["all"]:
    idx = np.concatenate([PARTS[q] for q in PART_DONE]) if p_ == "all" else PARTS[p_]
    for m in MODEL_KEYS:
        pr = PRED[m][idx]
        if not np.isfinite(pr).all():
            MISSING.append((p_, m)); continue
        q = detailed_metrics(y[idx], pr)
        ci_a = 100 * group_bootstrap_ci(y[idx], pr, groups[idx], acc_fn)
        ci_u = group_bootstrap_ci(y[idx], pr, groups[idx], roc_auc_score)
        ys_, ps_ = subject_level(idx, pr)
        qs = detailed_metrics(ys_, ps_)
        r = dict(part=p_ if p_ == "all" else p_ + 1, model=m, name=MODEL_NAME[m], family=MODEL_FAMILY[m], **q,
                 acc_lo=ci_a[0], acc_hi=ci_a[1], auc_lo=ci_u[0], auc_hi=ci_u[1],
                 subjects=f"{len(ys_)} ({int((ys_ == 0).sum())}/{int((ys_ == 1).sum())})", subj_n=len(ys_),
                 subj_accuracy=qs["accuracy"], subj_sensitivity=qs["recall"], subj_specificity=qs["specificity"],
                 subj_auc=qs["auc"], subj_f1=qs["f1"])
        if p_ != "all":
            tt = T_DF[(T_DF.part == p_ + 1) & (T_DF.model == m)].seconds
            r.update(time_per_fold_s=float(tt.mean()) if len(tt) else np.nan, time_total_s=float(tt.sum()) if len(tt) else np.nan)
            if PART_MODE == "independent":       # results of every inner fold of the part
                fm = []
                for k_ in sorted(np.unique(FOLD_OF[idx])):
                    sel = idx[FOLD_OF[idx] == k_]
                    qf = detailed_metrics(y[sel], PRED[m][sel])
                    ysf, psf = subject_level(sel, PRED[m][sel])
                    fm.append(qf)
                    fold_rows.append(dict(part=p_ + 1, fold=int(k_) + 1, model=m, name=MODEL_NAME[m],
                                          test_subjects=len(ysf), **qf, subj_accuracy=_acc(ysf, psf), subj_auc=_auc(ysf, psf)))
                fa, fu = np.array([f_["accuracy"] for f_ in fm]), np.array([f_["auc"] for f_ in fm])
                r.update(inner_acc_mean=np.nanmean(fa), inner_acc_sd=np.nanstd(fa, ddof=1) if len(fa) > 1 else np.nan,
                         inner_auc_mean=np.nanmean(fu), inner_auc_sd=np.nanstd(fu, ddof=1) if len(fu) > 1 else np.nan)
        rows.append(r)
RES = pd.DataFrame(rows)
FOLD_RES = pd.DataFrame(fold_rows)
RES_P = RES[RES.part != "all"].copy()
RES_P["part"] = RES_P.part.astype(int)
RES_ALL = RES[RES.part == "all"].set_index("model")
# ranks within each part (1 = best), by AUC and by accuracy
RES_P["rank_auc"] = RES_P.groupby("part").auc.rank(ascending=False, method="min")
RES_P["rank_acc"] = RES_P.groupby("part").accuracy.rank(ascending=False, method="min")
RES_P.to_csv(os.path.join(OUT_DIR, "tables", "all_metrics_per_part_and_classifier.csv"), index=False)
RES_ALL.reset_index().to_csv(os.path.join(OUT_DIR, "tables", "all_metrics_pooled_over_parts.csv"), index=False)
if len(FOLD_RES):
    FOLD_RES.to_csv(os.path.join(OUT_DIR, "tables", "all_metrics_per_inner_fold.csv"), index=False)
if MISSING:
    print("Missing predictions (classifier failed or was not run) for:", MISSING[:10])
print(f"Metrics computed for {len(PART_DONE)} part(s) x {len(MODEL_KEYS)} classifiers "
      f"({N_BOOTSTRAP} subject bootstrap resamples per CI, {time.time() - t0:.0f} s)")


# ---- formatting helpers ------------------------------------------------------------------------------------
def f_(v, nd):
    return "n/a" if v is None or not np.isfinite(v) else f"{v:.{nd}f}"


def msd(v, nd):
    v = pd.Series(v, dtype=float).dropna()
    if not len(v):
        return "n/a"
    return f"{v.mean():.{nd}f} ± {v.std(ddof=1):.{nd}f}" if len(v) > 1 else f"{v.mean():.{nd}f}"


def mrange(v, nd):
    v = pd.Series(v, dtype=float).dropna()
    return "n/a" if not len(v) else f"{v.min():.{nd}f} – {v.max():.{nd}f}"


# (row label, column in RES, decimals, kind) of the per-classifier detail table
DETAIL_ROWS = [("Test subjects (CN/AD)", "subjects", 0, "text"), ("Test images", "n", 0, "num"),
               ("Accuracy (%)", "accuracy", 1, "num"), ("  95% CI (subject bootstrap)", "acc", 1, "ci"),
               ("Balanced accuracy (%)", "balanced_accuracy", 1, "num"), ("Precision / PPV (%)", "precision", 1, "num"),
               ("Sensitivity / recall (%)", "recall", 1, "num"), ("Specificity (%)", "specificity", 1, "num"),
               ("NPV (%)", "npv", 1, "num"), ("F1-score (%)", "f1", 1, "num"), ("MCC", "mcc", 3, "num"),
               ("Cohen's κ", "kappa", 3, "num"), ("AUC", "auc", 3, "num"), ("  95% CI (subject bootstrap)", "auc", 3, "ci"),
               ("Brier score", "brier", 3, "num"), ("Log loss", "log_loss", 3, "num"),
               ("TP (AD → AD)", "tp", 0, "num"), ("FN (AD → CN)", "fn", 0, "num"), ("TN (CN → CN)", "tn", 0, "num"),
               ("FP (CN → AD)", "fp", 0, "num"), ("Subject-level accuracy (%)", "subj_accuracy", 1, "num"),
               ("Subject-level sensitivity (%)", "subj_sensitivity", 1, "num"),
               ("Subject-level specificity (%)", "subj_specificity", 1, "num"), ("Subject-level AUC", "subj_auc", 3, "num")]
if PART_MODE == "independent":
    DETAIL_ROWS += [("Inner-fold accuracy (%), mean ± SD", "inner_acc", 1, "inner"),
                    ("Inner-fold AUC, mean ± SD", "inner_auc", 3, "inner")]
DETAIL_ROWS += [("Training time per fold (s)", "time_per_fold_s", 1, "num")]


def model_table(m):
    """Rows = metrics; columns = every part, mean ± SD over parts, min – max, all parts pooled."""
    r = RES_P[RES_P.model == m].set_index("part")
    pool = RES_ALL.loc[m] if m in RES_ALL.index else None
    out = []
    for lab, col, nd, kind in DETAIL_ROWS:
        row = {"Metric": lab}
        for p_ in PART_DONE:
            rr = r.loc[p_ + 1] if (p_ + 1) in r.index else None
            if rr is None:
                row[f"Part {p_ + 1}"] = "n/a"
            elif kind == "text":
                row[f"Part {p_ + 1}"] = rr[col]
            elif kind == "ci":
                row[f"Part {p_ + 1}"] = f"{f_(rr[col + '_lo'], nd)} – {f_(rr[col + '_hi'], nd)}"
            elif kind == "inner":
                row[f"Part {p_ + 1}"] = f"{f_(rr[col + '_mean'], nd)} ± {f_(rr[col + '_sd'], nd)}"
            else:
                row[f"Part {p_ + 1}"] = f_(rr[col], nd)
        vals = r[col] if (kind == "num" and col in r) else None
        row["Mean ± SD over parts"] = msd(vals, nd) if vals is not None else ""
        row["Min – max"] = mrange(vals, nd) if vals is not None else ""
        if pool is None or kind == "inner" or col not in pool and kind != "ci":
            row["All parts pooled"] = ""
        elif kind == "ci":
            row["All parts pooled"] = f"{f_(pool[col + '_lo'], nd)} – {f_(pool[col + '_hi'], nd)}"
        elif kind == "text":
            row["All parts pooled"] = pool[col]
        else:
            row["All parts pooled"] = f_(pool[col], nd) if col in pool and pd.notna(pool[col]) else ""
        out.append(row)
    return pd.DataFrame(out)


def part_table(p_):
    """Rows = classifiers; columns = all metrics of this part."""
    r = RES_P[RES_P.part == p_ + 1].set_index("model")
    out = []
    for i, m in enumerate(MODEL_KEYS):
        if m not in r.index:
            continue
        q = r.loc[m]
        row = {"#": i + 1, "Classifier": MODEL_NAME[m], "Family": MODEL_FAMILY[m],
               "Accuracy (%)": f_(q.accuracy, 1), "Acc 95% CI": f"{f_(q.acc_lo, 1)}–{f_(q.acc_hi, 1)}",
               "Balanced acc (%)": f_(q.balanced_accuracy, 1), "Precision (%)": f_(q.precision, 1),
               "Sensitivity (%)": f_(q.recall, 1), "Specificity (%)": f_(q.specificity, 1), "NPV (%)": f_(q.npv, 1),
               "F1 (%)": f_(q.f1, 1), "MCC": f_(q.mcc, 3), "κ": f_(q.kappa, 3), "AUC": f_(q.auc, 3),
               "AUC 95% CI": f"{f_(q.auc_lo, 3)}–{f_(q.auc_hi, 3)}", "Brier": f_(q.brier, 3),
               "TP/FN/TN/FP": f"{q.tp}/{q.fn}/{q.tn}/{q.fp}", "Subject acc (%)": f_(q.subj_accuracy, 1),
               "Subject AUC": f_(q.subj_auc, 3)}
        if PART_MODE == "independent":
            row["Inner-fold acc (%)"] = f"{f_(q.inner_acc_mean, 1)} ± {f_(q.inner_acc_sd, 1)}"
        row.update({"Time / fold (s)": f_(q.time_per_fold_s, 1), "Rank (AUC)": int(q.rank_auc) if np.isfinite(q.rank_auc) else ""})
        out.append(row)
    return pd.DataFrame(out)


PART_TABLES = {p_: part_table(p_) for p_ in PART_DONE}
MODEL_TABLES = {m: model_table(m) for m in MODEL_KEYS if m in set(RES_P.model)}
for p_, t in PART_TABLES.items():
    t.to_csv(os.path.join(OUT_DIR, "tables", f"part{p_ + 1}_all_classifiers.csv"), index=False)
for i, (m, t) in enumerate(MODEL_TABLES.items()):
    t.to_csv(os.path.join(OUT_DIR, "tables", f"classifier{i + 1:02d}_{re.sub(r'[^A-Za-z0-9]+', '_', m)}_by_part.csv"),
             index=False)


## 11. Detailed results by part — all classifiers in each part
One table per part, one row per classifier. *Acc/AUC 95% CI*: subject bootstrap within the part. *Rank (AUC)*: 1 = highest AUC in that part (descriptive only).

In [ ]:
pd.set_option("display.max_columns", 60, "display.width", 250, "display.max_colwidth", 60)
for p_ in PART_DONE:
    pidx = PARTS[p_]
    best = RES_P[RES_P.part == p_ + 1].sort_values(["auc", "accuracy"], ascending=False).iloc[0]
    display(Markdown(
        f"### Part {p_ + 1} — {len(set(groups[pidx]))} subjects ({len(set(groups[pidx][y[pidx] == 0]))} CN / "
        f"{len(set(groups[pidx][y[pidx] == 1]))} AD), {len(pidx)} images\n"
        + ("Every classifier was trained and tested **inside this part only** "
           f"({TASKS[0]['n_folds']}-fold subject-level CV; each image tested once). "
           if PART_MODE == "independent" else
           f"Tested on this part; trained on the other {N_PARTS_USED - 1} parts. ")
        + f"Highest AUC in this part (descriptive): **{best['name']}** (AUC {best.auc:.3f}, accuracy {best.accuracy:.1f}%)."))
    display(PART_TABLES[p_])


## 12. Detailed results for each classifier
For every classifier: what it uses, a one-paragraph summary, a table with **one column per part** plus mean ± SD, min–max and all parts pooled, and the scikit-learn classification report (per-class precision, recall, F1 and support) over all parts.

In [ ]:
CLS_TEXT = {}
idx_done = np.concatenate([PARTS[q] for q in PART_DONE])
for i, m in enumerate(MODEL_KEYS):
    if m not in MODEL_TABLES:
        display(Markdown(f"### {i + 1}. {MODEL_NAME[m]}\nNo complete predictions (see the error list in Section 10)."))
        continue
    r = RES_P[RES_P.model == m].sort_values("part")
    best_p, worst_p = r.loc[r.accuracy.idxmax()], r.loc[r.accuracy.idxmin()]
    n_sig = int((r.auc_lo > 0.5).sum())
    CLS_TEXT[m] = (
        f"{MODEL_NAME[m]}: accuracy {msd(r.accuracy, 1)}% over {len(r)} part(s) "
        + (f"(range {mrange(r.accuracy, 1)}%; highest in Part {int(best_p.part)}, lowest in Part {int(worst_p.part)}), "
           if r.accuracy.nunique() > 1 else "(the same in every part), ")
        + f"AUC {msd(r.auc, 3)}, sensitivity {msd(r.recall, 1)}%, specificity {msd(r.specificity, 1)}%, "
        f"F1 {msd(r.f1, 1)}%, subject-level accuracy {msd(r.subj_accuracy, 1)}%. "
        f"The 95% CI of the AUC lies above 0.5 in {n_sig} of {len(r)} part(s); mean rank by AUC "
        f"{r.rank_auc.mean():.1f} of {len(MODEL_TABLES)}.")
    display(Markdown(f"### {i + 1}. {MODEL_NAME[m]}\n*Family:* {MODEL_FAMILY[m]} · *Input / settings:* {MODEL_INPUT.get(m, '')}\n\n"
                     + CLS_TEXT[m]))
    display(MODEL_TABLES[m])
    pr = PRED[m][idx_done]
    ok_ = np.isfinite(pr)
    print(f"Classification report — {MODEL_NAME[m]}, all {len(PART_DONE)} part(s) pooled "
          f"({int(ok_.sum())} test images, threshold 0.5):")
    print(classification_report(y[idx_done][ok_], (pr[ok_] >= 0.5).astype(int), labels=[0, 1],
                                target_names=["CN", "AD"], digits=3, zero_division=0))


## 13. Summary across parts: comparison, consistency and tests
- **Summary table**: mean ± SD over parts for the main metrics, min–max, mean rank (by AUC and accuracy), and in how many parts the AUC's 95% CI lies above chance.
- **Classifier × part matrices** of accuracy and AUC, with the best classifier of each part (descriptive).
- **Friedman test**: whether the classifiers' ranks differ consistently over parts (and over inner folds). With few parts it has little power; a non-significant result does not show that classifiers are equal.

In [ ]:
KEYS_SUM = [("accuracy", "Accuracy (%)", 1), ("auc", "AUC", 3), ("balanced_accuracy", "Balanced acc (%)", 1),
            ("recall", "Sensitivity (%)", 1), ("specificity", "Specificity (%)", 1), ("f1", "F1 (%)", 1),
            ("mcc", "MCC", 3), ("subj_accuracy", "Subject acc (%)", 1)]
agg = RES_P.groupby("model")
order_sum = (pd.DataFrame({"auc": agg.auc.mean(), "acc": agg.accuracy.mean()})
             .sort_values(["auc", "acc"], ascending=False).index.tolist())
rows = []
for rk, m in enumerate(order_sum, 1):
    r = RES_P[RES_P.model == m]
    row = {"Rank (mean AUC)": rk, "Classifier": MODEL_NAME[m], "Family": MODEL_FAMILY[m]}
    for col, lab, nd in KEYS_SUM:
        row[f"{lab} mean ± SD"] = msd(r[col], nd)
        if col in ("accuracy", "auc"):
            row[f"{lab} min – max"] = mrange(r[col], nd)
    row.update({"Mean rank by AUC": round(float(r.rank_auc.mean()), 2), "Mean rank by accuracy": round(float(r.rank_acc.mean()), 2),
                "Parts with AUC CI > 0.5": f"{int((r.auc_lo > 0.5).sum())} of {len(r)}",
                "Pooled accuracy (%)": f_(RES_ALL.loc[m, "accuracy"], 1) if m in RES_ALL.index else "n/a",
                "Pooled AUC": f_(RES_ALL.loc[m, "auc"], 3) if m in RES_ALL.index else "n/a",
                "Time per part (s)": f_(r.time_total_s.mean(), 1)})
    rows.append(row)
SUMMARY = pd.DataFrame(rows)
SUMMARY.to_csv(os.path.join(OUT_DIR, "tables", "summary_all_classifiers_across_parts.csv"), index=False)
display(Markdown(f"**Summary across {len(PART_DONE)} part(s)** — sorted by mean AUC. The order is descriptive: it was "
                 "computed after testing and must not be used to pick a model."))
display(SUMMARY)


def by_part_matrix(col, nd):
    piv = RES_P.pivot(index="model", columns="part", values=col).reindex([m for m in MODEL_KEYS if m in set(RES_P.model)])
    num = piv.copy()
    num.columns = [f"Part {c}" for c in piv.columns]
    num.insert(0, "Classifier", [MODEL_NAME[m] for m in piv.index])
    num["Mean"], num["SD"] = piv.mean(axis=1).values, piv.std(axis=1, ddof=1).values
    num = num.reset_index(drop=True)
    best = {"Classifier": "Highest in part (descriptive)"}
    for c in piv.columns:
        v = piv[c]
        best[f"Part {c}"] = ", ".join(MODEL_NAME[m] for m in v.index[v == v.max()])
    show = num.copy()
    for c in show.columns[1:]:
        show[c] = [f"{v:.{nd}f}" if np.isfinite(v) else "n/a" for v in num[c]]
    return num.round(nd), pd.concat([show, pd.DataFrame([best])], ignore_index=True).fillna("")


ACC_MAT, ACC_MAT_SHOW = by_part_matrix("accuracy", 1)
AUC_MAT, AUC_MAT_SHOW = by_part_matrix("auc", 3)
ACC_MAT.to_csv(os.path.join(OUT_DIR, "tables", "accuracy_by_part.csv"), index=False)
AUC_MAT.to_csv(os.path.join(OUT_DIR, "tables", "auc_by_part.csv"), index=False)
print("Accuracy (%) of every classifier in every part:"); display(ACC_MAT_SHOW)
print("AUC of every classifier in every part:"); display(AUC_MAT_SHOW)

# Friedman test: do the classifiers differ consistently across parts (and, in "independent" mode, across inner folds)?
FRIEDMAN = {}
models_ok = [m for m in MODEL_KEYS if m in set(RES_P.model)]
for lab, tab, block in (("parts", RES_P, ["part"]), ("inner folds", FOLD_RES, ["part", "fold"])):
    if not len(tab) or len(models_ok) < 3:
        continue
    for col in ("accuracy", "auc"):
        mat = tab.pivot_table(index=block, columns="model", values=col)[models_ok].dropna()
        if len(mat) >= 2:
            st = stats.friedmanchisquare(*[mat[m].values for m in models_ok])
            FRIEDMAN[(lab, col)] = (len(mat), float(st.statistic), float(st.pvalue))
            print(f"Friedman test over {len(mat)} {lab} ({len(models_ok)} classifiers, {col}): "
                  f"chi2 = {st.statistic:.2f}, p = {st.pvalue:.3g}")
if len(PART_DONE) < 3:
    print("NOTE: with fewer than 3 parts, the spread over parts (SD) is a weak estimate.")


## 14. Figures
- **P1** heat map: accuracy and AUC of every classifier in every part (diverging colours around chance).
- **P2** mean ± SD over parts, with each part's value as a dot.
- **P3** ROC curves of every classifier, one curve per part.
- **P4** confusion matrices of every classifier, summed over the parts.
- **P5** CNN fine-tuning curves of every training run.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

INK, INK2, MUTED, GRID = "#0b0b0b", "#52514e", "#898781", "#e1e0d9"
DIVERGING = LinearSegmentedColormap.from_list("chance_div", ["#c43a3a", "#e34948", "#f0efec", "#2a78d6", "#1c5cab"])
SHOW = [m for m in MODEL_KEYS if m in set(RES_P.model)]
P_SHOW = [p_ + 1 for p_ in PART_DONE]

# ---- Figure P1: accuracy and AUC of every classifier in every part (diverging around chance) ----
fig, axes = plt.subplots(1, 2, figsize=(4.6 + 2.2 * len(P_SHOW), 0.36 * len(SHOW) + 1.8), sharey=True)
for ax, (col, lab, chance, nd) in zip(axes, (("accuracy", "Accuracy (%)", 50.0, 1), ("auc", "AUC", 0.5, 3))):
    piv = RES_P.pivot(index="model", columns="part", values=col).reindex(SHOW)[P_SHOW]
    M = np.column_stack([piv.values, piv.mean(axis=1).values])
    half = max(float(np.nanmax(np.abs(M - chance))), 1e-6)
    ax.imshow(M, cmap=DIVERGING, norm=TwoSlopeNorm(chance, chance - half, chance + half), aspect="auto")
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            if np.isfinite(M[i, j]):
                ax.text(j, i, f"{M[i, j]:.{1 if nd == 1 else 2}f}", ha="center", va="center", fontsize=7.5,
                        color="white" if abs(M[i, j] - chance) > 0.65 * half else INK,
                        fontweight="bold" if j == M.shape[1] - 1 else "normal")
    ax.set_xticks(range(M.shape[1])); ax.set_xticklabels([f"Part {p}" for p in P_SHOW] + ["Mean"], fontsize=8)
    ax.axvline(M.shape[1] - 1.5, color="white", lw=2.5)
    ax.set_title(f"{lab} (red = below chance, blue = above)", fontsize=9); ax.grid(False)
axes[0].set_yticks(range(len(SHOW))); axes[0].set_yticklabels([MODEL_NAME[m] for m in SHOW], fontsize=8)
fig.suptitle(f"Figure P1. Every classifier in every part ({PART_MODE} parts; chance = 50% / 0.5)", fontsize=10)
fig.tight_layout(); FIGS["figP1"] = savefig(fig, "figP1_heatmap_classifier_by_part")

# ---- Figure P2: mean ± SD over parts, with the value of every part ----
order_p2 = RES_P.groupby("model").auc.mean().reindex(SHOW).sort_values().index.tolist()
fig, axes = plt.subplots(1, 2, figsize=(12, 0.36 * len(order_p2) + 1.9), sharey=True)
for ax, (col, lab, chance, lim) in zip(axes, (("accuracy", "Accuracy (%)", 50, (0, 100)), ("auc", "AUC", 0.5, (0, 1)))):
    g = RES_P.groupby("model")[col]
    mu, sd = g.mean().reindex(order_p2), g.std(ddof=1).reindex(order_p2).fillna(0)
    ax.barh(range(len(order_p2)), mu.values, height=0.6, color=[FAMILY_COLORS[MODEL_FAMILY[m]] for m in order_p2],
            xerr=sd.values, error_kw=dict(ecolor=INK, elinewidth=0.9, capsize=2))
    for i, m in enumerate(order_p2):
        v = RES_P[RES_P.model == m][col].values
        ax.scatter(v, np.full(len(v), i), s=12, color="white", edgecolor=INK, linewidth=0.7, zorder=3)
    ax.axvline(chance, color=INK, ls="--", lw=1)
    ax.set(xlim=lim, xlabel=lab)
    ax.grid(axis="y", visible=False)
axes[0].set_yticks(range(len(order_p2))); axes[0].set_yticklabels([MODEL_NAME[m] for m in order_p2], fontsize=8)
fams = [f for f in FAMILY_COLORS if f in {MODEL_FAMILY[m] for m in order_p2}]
fig.legend([plt.Rectangle((0, 0), 1, 1, color=FAMILY_COLORS[f]) for f in fams], fams, loc="lower center",
           ncol=len(fams), fontsize=8, bbox_to_anchor=(0.5, -0.02))
fig.suptitle(f"Figure P2. Classifiers across {len(P_SHOW)} parts — bar = mean, whisker = SD, dots = individual parts, dashed line = chance", fontsize=10)
fig.tight_layout(rect=(0, 0.05, 1, 0.97)); FIGS["figP2"] = savefig(fig, "figP2_mean_sd_over_parts")

# ---- Figure P3: ROC curve of every classifier in every part ----
ncol = 5
nrow = int(np.ceil(len(SHOW) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(3.0 * ncol, 3.05 * nrow), squeeze=False)
for ax, m in zip(axes.ravel(), SHOW):
    for j, p_ in enumerate(PART_DONE):
        idx = PARTS[p_]
        if np.isfinite(PRED[m][idx]).all() and len(np.unique(y[idx])) == 2:
            fpr, tpr, _ = roc_curve(y[idx], PRED[m][idx])
            ax.plot(fpr, tpr, color=PART_COLORS[p_ % len(PART_COLORS)], lw=1.6, label=f"Part {p_ + 1}")
    ax.plot([0, 1], [0, 1], ls="--", color=MUTED, lw=1)
    ax.set(xlim=(0, 1), ylim=(0, 1.01), xticks=[0, 0.5, 1], yticks=[0, 0.5, 1])
    ax.set_title(f"{MODEL_NAME[m]}\nAUC {msd(RES_P[RES_P.model == m].auc, 3)}", fontsize=7.5)
    ax.tick_params(labelsize=7); ax.set_aspect("equal")
for ax in axes.ravel()[len(SHOW):]:
    ax.axis("off")
for ax in axes[:, 0]:
    ax.set_ylabel("Sensitivity", fontsize=8)
for ax in axes[-1, :]:
    ax.set_xlabel("1 − specificity", fontsize=8)
h_, l_ = axes.ravel()[0].get_legend_handles_labels()
fig.legend(h_, l_, loc="lower center", ncol=len(l_), fontsize=8, bbox_to_anchor=(0.5, -0.01))
fig.suptitle("Figure P3. ROC curves of every classifier — one curve per part (test predictions)", fontsize=10)
fig.tight_layout(rect=(0, 0.03, 1, 0.98)); FIGS["figP3"] = savefig(fig, "figP3_roc_every_classifier_by_part")

# ---- Figure P4: confusion matrices of every classifier, summed over the parts ----
fig, axes = plt.subplots(nrow, ncol, figsize=(2.55 * ncol, 2.65 * nrow), squeeze=False)
for ax, m in zip(axes.ravel(), SHOW):
    q = RES_ALL.loc[m]
    cm = np.array([[q.tn, q.fp], [q.fn, q.tp]], dtype=float)
    ax.imshow(cm / cm.sum(1, keepdims=True), cmap=LinearSegmentedColormap.from_list("b", ["#f0efec", "#2a78d6", "#104281"]),
              vmin=0, vmax=1)
    for i in range(2):
        for j in range(2):
            frac = cm[i, j] / cm[i].sum()
            ax.text(j, i, f"{int(cm[i, j])}\n{100 * frac:.0f}%", ha="center", va="center", fontsize=8,
                    color="white" if frac > 0.6 else INK)
    ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=["CN", "AD"], yticklabels=["CN", "AD"])
    ax.set_title(MODEL_NAME[m], fontsize=7.5); ax.tick_params(labelsize=7); ax.grid(False)
for ax in axes.ravel()[len(SHOW):]:
    ax.axis("off")
for ax in axes[:, 0]:
    ax.set_ylabel("True", fontsize=8)
for ax in axes[-1, :]:
    ax.set_xlabel("Predicted", fontsize=8)
fig.suptitle(f"Figure P4. Confusion matrices (counts and row %), summed over {len(P_SHOW)} part(s)", fontsize=10)
fig.tight_layout(); FIGS["figP4"] = savefig(fig, "figP4_confusion_matrices_every_classifier")

# ---- Figure P5: CNN training curves of every training run, coloured by part ----
if HIST_ALL:
    H = pd.concat(HIST_ALL)
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
    for (pp, ff), h in H.groupby(["part", "fold"]):
        c = PART_COLORS[(int(pp) - 1) % len(PART_COLORS)]
        axes[0].plot(h.epoch, h.train_loss, color=c, lw=1, alpha=0.8)
        axes[1].plot(h.epoch, h.val_loss, color=c, lw=1, alpha=0.8)
        axes[2].plot(h.epoch, h.val_auc, color=c, lw=1, alpha=0.8)
    for ax, t in zip(axes, ("Training loss", "Validation loss", "Validation subject-level AUC")):
        ax.set(xlabel="Epoch", title=t); ax.xaxis.set_major_locator(matplotlib.ticker.MaxNLocator(integer=True))
    axes[2].axhline(0.5, color=INK, ls="--", lw=1)
    fig.legend([plt.Line2D([], [], color=PART_COLORS[(p - 1) % len(PART_COLORS)]) for p in P_SHOW],
               [f"Part {p}" for p in P_SHOW], loc="lower center", ncol=len(P_SHOW), fontsize=8, bbox_to_anchor=(0.5, -0.04))
    fig.suptitle("Figure P5. CNN fine-tuning curves (one line per training run)", fontsize=10)
    fig.tight_layout(rect=(0, 0.06, 1, 0.95)); FIGS["figP5"] = savefig(fig, "figP5_cnn_training_curves")


## 15. Exports
`tables/results_by_part_and_classifier.xlsx` (sheets: Summary, Accuracy by part, AUC by part, one sheet per part, one per classifier, long tables, splits and subjects), the same tables as CSV, all test predictions (`test_predictions_all_parts.csv`), a text summary (`results_summary.md`), the run configuration and the per-part result files in `results/`; everything zipped.

In [ ]:
# ---- all test predictions in one file (each image tested exactly once) ----
PRED_DF = pd.DataFrame({"rel": df.rel, "subject": df.subject, "cdr": df.cdr_folder, "label": y, "part": PART_OF + 1,
                        "inner_fold": FOLD_OF + 1, **{MODEL_NAME[m]: PRED[m] for m in MODEL_KEYS}})
PRED_DF = PRED_DF[PRED_DF.part.isin([p_ + 1 for p_ in PART_DONE])]
PRED_DF.to_csv(os.path.join(OUT_DIR, "tables", "test_predictions_all_parts.csv"), index=False)


# ---- Excel workbook: summary, one sheet per part, one sheet per classifier, long tables ----
def sheet_name(s, used):
    s = re.sub(r"[\[\]\*\?/\\:]", "", s)[:31] or "sheet"
    base, k = s, 2
    while s in used:
        s = f"{base[:28]}~{k}"; k += 1
    used.add(s)
    return s


def save_workbook():
    path = os.path.join(OUT_DIR, "tables", "results_by_part_and_classifier.xlsx")
    used, sheets = set(), {"Summary": SUMMARY, "Accuracy by part": ACC_MAT, "AUC by part": AUC_MAT,
                           **{f"Part {p_ + 1}": PART_TABLES[p_] for p_ in PART_DONE},
                           **{f"{i + 1:02d} {MODEL_NAME[m]}": MODEL_TABLES[m] for i, m in enumerate(MODEL_KEYS)
                              if m in MODEL_TABLES},
                           "All metrics (long)": RES_P.round(4), "Pooled over parts": RES_ALL.reset_index().round(4),
                           "Parts": PART_TABLE, "Splits": TASK_TABLE, "Subjects per part": SUBJECT_PARTS}
    if len(FOLD_RES):
        sheets["Inner folds (long)"] = FOLD_RES.round(4)
    try:
        with pd.ExcelWriter(path) as xw:
            for name, t in sheets.items():
                t.to_excel(xw, sheet_name=sheet_name(name, used), index=False)
        print("Saved", path, f"({len(sheets)} sheets)")
    except Exception as e:
        print(f"Excel export skipped ({type(e).__name__}: {e}); the CSV files in tables/ contain the same tables.")


save_workbook()

# ---- short text summary (every number computed from this run) ----
lines = [f"# Small-data, multi-part results ({datetime.date.today().isoformat()})", "",
         f"Data: {DATASET_NAME}. Small dataset: {df.subject.nunique()} subjects ({int((y == 0).sum())} CN / "
         f"{int((y == 1).sum())} AD images from {df[df.label == 0].subject.nunique()} CN and "
         f"{df[df.label == 1].subject.nunique()} AD subjects), {len(df)} images, {SLICES_PER_SUBJECT} slices per subject.",
         f"Parts: {N_PARTS_USED} disjoint, class-balanced parts of subjects; mode '{PART_MODE}'"
         + (f" ({TASKS[0]['n_folds']}-fold subject-level CV inside each part)." if PART_MODE == "independent" else
            " (each part tested by models trained on the other parts).")
         + f" Results available for part(s) {', '.join(str(p_ + 1) for p_ in PART_DONE)}.",
         "No subject appears in more than one of training, validation and test data (checked by assertion).", "",
         "## Per classifier (mean ± SD over parts)", ""]
lines += [f"- {CLS_TEXT[m]}" for m in order_sum if m in CLS_TEXT]
lines += ["", "## Statistical tests", ""]
lines += [f"- Friedman over {n} {lab} ({col}): chi2 = {s_:.2f}, p = {p:.3g}" for (lab, col), (n, s_, p) in FRIEDMAN.items()]
lines += ["", "With ~10 subjects per class in a part, a single part's accuracy can move by 5–10 points when one subject "
          "changes; read the mean ± SD over parts and the CIs, not the best part."]
RESULTS_MD = "\n".join(lines)
with open(os.path.join(OUT_DIR, "results_summary.md"), "w") as fh:
    fh.write(RESULTS_MD)
display(Markdown(RESULTS_MD))

with open(os.path.join(OUT_DIR, "run_config.json"), "w") as fh:
    json.dump({**{k: v for k, v in _sig_cfg.items() if k != "images"}, "signature": SIGNATURE, "env": ENV,
               "circuit": CIRCUIT_INFO, "qiskit_check": {str(k): v for k, v in SIM_CHECK.items()},
               "parts_done": [p_ + 1 for p_ in PART_DONE], "n_images": len(df)}, fh, indent=1, default=str)

zip_path = os.path.join(os.path.dirname(OUT_DIR), os.path.basename(OUT_DIR) + ".zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for rt, _, files in os.walk(OUT_DIR):
        if os.path.basename(rt) == "cache":
            continue
        for f in files:
            full = os.path.join(rt, f)
            zf.write(full, os.path.relpath(full, os.path.dirname(OUT_DIR)))
print("All outputs:", OUT_DIR, "| zipped:", zip_path)
